# 01 – Data audit
Goal: identify the data-quality issues that most affect business metrics, document how each is treated, and measure the impact.
Rule: exercise_* tables are never modified.

#### Questions to ask the table:
##### 1. What is it? (Understand)
- **What does one row represent?**
- **What’s the unique key?**
- **How does the table join to the others?**
- **What period does it cover?**

##### 2. Is anything repeated? (Uniqueness)
- **Does the tabale has duplicate rows or keys?**
- **Are they exact copies (safe to drop) or conflicting versions (need a rule)?**

##### 3. Is anything missing? (Completeness)
- **Does it have Nulls and blanks?**
- **Are they, especially, in keys and fields your metrics use?**
- **Are they missing, or not applicable?**

##### 4. Are the values possible? (Validity)
- **Out-of-range numbers?**
- **Placeholder values?**
- **Unexpected categories?**

##### 5.Does it all connect? (Referential integrity)
- **Do IDs point to records that don’t exist?**

##### 6. Do the dates make sense? (Time logic and freshness)
- **Did tings happened in the wrong order, before something existed, or tables ending at different times?**

##### 7. Is it real? (Test data and plausibility)
- **Are there test records, bots, extreme outliers?**

##### For every issue you find, record five things
1. What the issue is
2. How many rows it affects (count and %)
3. Which metric it touches
4. What you did: fix, deduplicate, quarantine, flag, or leave it with a caveat
5. The impact on the number a business user would see
6. What makes an issue “matter most”

##### When does it matter the most?
Size × whether it touches a core metric.

In [0]:
%sql
SHOW TABLES IN workspace.scribd_exercise;

database,tableName,isTemporary
scribd_exercise,exercise_businesses,false
scribd_exercise,exercise_content_items,false
scribd_exercise,exercise_creators,false
scribd_exercise,exercise_engagement_events,false
scribd_exercise,exercise_subscriptions,false
scribd_exercise,exesrcise_users,false


In [0]:
%python
from pyspark.sql import functions as F

schema = "workspace.scribd_exercise"
tables = [r.tableName for r in spark.sql(f"SHOW TABLES IN {schema}").collect()
          if r.tableName.startswith("exercise_")]

results = []
for t in tables:
    df = spark.table(f"{schema}.{t}")
    aggs = [F.count(F.lit(1)).alias("total")]
    for i, c in enumerate(df.columns):
        col = F.col(f"`{c}`")
        aggs += [
            F.sum(col.isNull().cast("int")).alias(f"nulls_{i}"),
            F.sum((F.trim(col.cast("string")) == "").cast("int")).alias(f"blanks_{i}"),
            F.countDistinct(col).alias(f"distinct_{i}"),
            F.min(col).cast("string").alias(f"min_{i}"),
            F.max(col).cast("string").alias(f"max_{i}"),
        ]
    r = df.agg(*aggs).collect()[0]
    total = r["total"]
    for i, c in enumerate(df.columns):
        nulls = r[f"nulls_{i}"] or 0
        blanks = r[f"blanks_{i}"] or 0
        results.append((t, c, str(df.schema[c].dataType), total, nulls, blanks,
                        round(100 * (nulls + blanks) / total, 2) if total else None,
                        r[f"distinct_{i}"], r[f"min_{i}"], r[f"max_{i}"]))

profile = spark.createDataFrame(results, ["table", "column", "type", "total_rows", "nulls",
                                          "blanks", "missing_pct", "distinct_count", "min", "max"])
display(profile.orderBy("table", F.desc("missing_pct")))

table,column,type,total_rows,nulls,blanks,missing_pct,distinct_count,min,max
exercise_businesses,business_id,StringType(),3,0,0,0.0,3,BIZ_1,BIZ_3
exercise_businesses,business_name,StringType(),3,0,0,0.0,3,GatherSquare Community (SYNTHETIC),PageGarden Premium (SYNTHETIC)
exercise_businesses,business_line,StringType(),3,0,0,0.0,3,consumer_community,premium_published_content
exercise_businesses,launch_date,DateType(),3,0,0,0.0,3,2021-01-01,2022-03-01
exercise_businesses,hq_planet,StringType(),3,0,0,0.0,3,Nearia,Wavanda
exercise_content_items,creator_id,StringType(),300,195,0,65.0,34,CR_0001,CR_0040
exercise_content_items,content_id,StringType(),300,0,0,0.0,300,CNT_00001,CNT_00300
exercise_content_items,business_id,StringType(),300,0,0,0.0,3,BIZ_1,BIZ_3
exercise_content_items,title,StringType(),300,0,0,0.0,300,Episode 10: Totally Original Content (Mock #276),[MOCK THREAD] Unpopular opinion: fake topic #90
exercise_content_items,category,StringType(),300,0,0,0.0,17,Ask the Community,Serialized Drama


# Businesses Table


In [0]:
%sql 
DESCRIBE TABLE workspace.scribd_exercise.exercise_businesses;

col_name,data_type,comment
business_id,string,null
business_name,string,null
business_line,string,null
launch_date,date,null
hq_planet,string,null
# Clustering Information,,
# col_name,data_type,comment


In [0]:
%sql 
SELECT * FROM workspace.scribd_exercise.exercise_businesses;

business_id,business_name,business_line,launch_date,hq_planet
BIZ_1,PageGarden Premium (SYNTHETIC),premium_published_content,2021-01-01,Terra Prime
BIZ_2,GatherSquare Community (SYNTHETIC),consumer_community,2021-06-15,Nearia
BIZ_3,MakerHarbor (SYNTHETIC),creator_content,2022-03-01,Wavanda


In [0]:
-- How much each business appears in other tables 
SELECT b.business_id,
b.business_line,

(SELECT COUNT(*) FROM workspace.scribd_exercise.exercise_content_items ci
WHERE ci.business_id = b.business_id) AS content_items,

(SELECT COUNT(*) FROM workspace.scribd_exercise.exercise_creators cr
WHERE cr.business_id = b.business_id) AS creators,

(SELECT COUNT(*) FROM workspace.scribd_exercise.exercise_subscriptions s
WHERE s.business_id = b.business_id) AS subscriptions,

(SELECT COUNT(*) FROM workspace.scribd_exercise.exercise_engagement_events e
JOIN workspace.scribd_exercise.exercise_content_items ci2 ON e.content_id = ci2.content_id
WHERE ci2.business_id = b.business_id) AS events_via_content

FROM workspace.scribd_exercise.exercise_businesses b
ORDER BY b.business_id;

business_id,business_line,content_items,creators,subscriptions,events_via_content
BIZ_1,premium_published_content,112,3,272,2078
BIZ_2,consumer_community,83,0,297,1627
BIZ_3,creator_content,105,37,274,1972


**Findings:** 
- Every business has content, subscriptions and events. Counts by business reconcile to each table's total (content 300, creators 40, subscriptions 843, events with content 5,677), 
- No record in any table carries an unknown business_id

In [0]:
-- AI suggested and generated 
-- Earliest activity per business vs. launch
WITH b AS (
  SELECT business_id, launch_date
  FROM workspace.scribd_exercise.exercise_businesses
)
SELECT b.business_id, b.launch_date, 'creators (join_date)' AS activity,
       MIN(cr.join_date) AS earliest,
       SUM(CASE WHEN cr.join_date < b.launch_date THEN 1 ELSE 0 END) AS before_launch,
       MAX(CASE WHEN cr.join_date < b.launch_date THEN DATEDIFF(b.launch_date, cr.join_date) END) AS max_days_before
FROM b JOIN workspace.scribd_exercise.exercise_creators cr ON cr.business_id = b.business_id
GROUP BY b.business_id, b.launch_date
UNION ALL
SELECT b.business_id, b.launch_date, 'subscriptions (start_date)',
       MIN(s.start_date),
       SUM(CASE WHEN s.start_date < b.launch_date THEN 1 ELSE 0 END),
       MAX(CASE WHEN s.start_date < b.launch_date THEN DATEDIFF(b.launch_date, s.start_date) END)
FROM b JOIN workspace.scribd_exercise.exercise_subscriptions s ON s.business_id = b.business_id
GROUP BY b.business_id, b.launch_date
UNION ALL
SELECT b.business_id, b.launch_date, 'events via content (event date)',
       MIN(CAST(e.event_timestamp AS DATE)),
       SUM(CASE WHEN CAST(e.event_timestamp AS DATE) < b.launch_date THEN 1 ELSE 0 END),
       MAX(CASE WHEN CAST(e.event_timestamp AS DATE) < b.launch_date THEN DATEDIFF(b.launch_date, CAST(e.event_timestamp AS DATE)) END)
FROM b
JOIN workspace.scribd_exercise.exercise_content_items ci ON ci.business_id = b.business_id
JOIN workspace.scribd_exercise.exercise_engagement_events e ON e.content_id = ci.content_id
GROUP BY b.business_id, b.launch_date
ORDER BY business_id, activity;

business_id,launch_date,activity,earliest,before_launch,max_days_before
BIZ_1,2021-01-01,creators (join_date),2020-12-07,1,25
BIZ_1,2021-01-01,events via content (event date),2021-01-09,0,null
BIZ_1,2021-01-01,subscriptions (start_date),2021-07-24,0,null
BIZ_2,2021-06-15,events via content (event date),2021-08-10,0,null
BIZ_2,2021-06-15,subscriptions (start_date),2021-12-01,0,null
BIZ_3,2022-03-01,creators (join_date),2022-02-05,1,24
BIZ_3,2022-03-01,events via content (event date),2021-03-21,3,345
BIZ_3,2022-03-01,subscriptions (start_date),2022-01-02,1,58


**Findings:**
- Creators: 2 joined before launch (BIZ_1: 25 days; BIZ_3: 24 days early). Plausible pre-launch recruiting **Recommendation:** keep + flag joined_before_launch.
- Subscriptions: 1 BIZ_3 subscription started 58 days before launch (possible pre-order or beta) **Recommendation:** keep + flag started_before_launch; revisit in subscriptions audit.
- Events: only the 3 BIZ_3 events already in issue #6. No new cases.

**Observation:** subscriptions start months after launch (BIZ_1: July 2021; BIZ_2: December 2021) while events start right away; subscription trends shouldn't be read as starting at launch.

# Content Items Table

In [0]:
DESCRIBE TABLE workspace.scribd_exercise.exercise_content_items;

col_name,data_type,comment
content_id,string,null
business_id,string,null
creator_id,string,null
title,string,null
category,string,null
content_type,string,null
publish_date,date,null
runtime_minutes_fake,bigint,null
# Clustering Information,,
# col_name,data_type,comment


In [0]:
SELECT * FROM workspace.scribd_exercise.exercise_content_items LIMIT 10;

content_id,business_id,creator_id,title,category,content_type,publish_date,runtime_minutes_fake
CNT_00001,BIZ_1,null,The Great Prairiefox (Mock Book #1),Sci-Fi,TEST_TYPE,2023-11-02,5
CNT_00002,BIZ_2,null,[MOCK THREAD] Rant: fake topic #2,Book Club Chatter,TEST_TYPE,2021-08-01,5
CNT_00003,BIZ_1,null,The Fake Diary (Mock Book #3),Cooking,ebook,2026-03-27,90
CNT_00004,BIZ_3,CR_0004,Episode 93: Totally Original Content (Mock #4),Poetry (Bad),audiobook,2026-09-14,45
CNT_00005,BIZ_1,null,The Fake Diary (Mock Book #5),Mystery,ebook,2025-05-25,25
CNT_00006,BIZ_3,CR_0040,Episode 4: Totally Original Content (Mock #6),Indie Fiction,poll,2026-02-21,2
CNT_00007,BIZ_1,null,The Secret Recipe (Mock Book #7),Sci-Fi,forum_post,2024-06-27,8
CNT_00008,BIZ_2,null,[MOCK THREAD] Anyone else fake topic #8,Book Club Chatter,video,2025-11-05,-1
CNT_00009,BIZ_2,null,[MOCK THREAD] PSA: fake topic #9,Ask the Community,audiobook,2022-03-18,180
CNT_00010,BIZ_1,null,The Great Diary (Mock Book #10),Romance,serialized_chapter,2023-08-28,45


#### Uniqueness - Is each content item listed only once? 

In [0]:
SELECT content_id,
COUNT(*) AS rows,
COUNT(DISTINCT title) AS titles,
COUNT(DISTINCT business_id) AS businesses,
COUNT(DISTINCT content_type) AS types,
COUNT(DISTINCT category) AS categories,
COUNT(DISTINCT publish_date) AS publish_dates, 
COUNT(DISTINCT runtime_minutes_fake) AS runtimes
FROM workspace.scribd_exercise.exercise_content_items
GROUP BY content_id
HAVING COUNT(*) > 1;

content_id,rows,titles,businesses,types,categories,publish_dates,runtimes


**Finding:** No duplicate content IDs. All rows have a unique content_id 

**Treatment:** None needed 

### Completeness - Which columns have missing values, and are they truly missing or not applicable?

In [0]:
SELECT 'content_id' AS column_name,
SUM(CASE WHEN content_id IS NULL THEN 1 ELSE 0 END) AS nulls,
SUM(CASE WHEN TRIM(content_id) = '' THEN 1 ELSE 0 END) AS blanks,
SUM(CASE WHEN LOWER(TRIM(content_id)) IN ('n/a','na','null','none','unknown','-') THEN 1 ELSE 0 END) AS placeholders
FROM workspace.scribd_exercise.exercise_content_items

UNION ALL

SELECT 'business_id',
SUM(CASE WHEN business_id IS NULL THEN 1 ELSE 0 END),
SUM(CASE WHEN TRIM(business_id) = '' THEN 1 ELSE 0 END),
SUM(CASE WHEN LOWER(TRIM(business_id)) IN ('n/a','na','null','none','unknown','-') THEN 1 ELSE 0 END)
FROM workspace.scribd_exercise.exercise_content_items

UNION ALL

SELECT 'creator_id',
SUM(CASE WHEN creator_id IS NULL THEN 1 ELSE 0 END),
SUM(CASE WHEN TRIM(creator_id) = '' THEN 1 ELSE 0 END),
SUM(CASE WHEN LOWER(TRIM(creator_id)) IN ('n/a','na','null','none','unknown','-') THEN 1 ELSE 0 END)
FROM workspace.scribd_exercise.exercise_content_items

UNION ALL

SELECT 'title',
SUM(CASE WHEN title IS NULL THEN 1 ELSE 0 END),
SUM(CASE WHEN TRIM(title) = '' THEN 1 ELSE 0 END),
SUM(CASE WHEN LOWER(TRIM(title)) IN ('n/a','na','null','none','unknown','-') THEN 1 ELSE 0 END)
FROM workspace.scribd_exercise.exercise_content_items

UNION ALL

SELECT 'category',
SUM(CASE WHEN category IS NULL THEN 1 ELSE 0 END),
SUM(CASE WHEN TRIM(category) = '' THEN 1 ELSE 0 END),
SUM(CASE WHEN LOWER(TRIM(category)) IN ('n/a','na','null','none','unknown','-') THEN 1 ELSE 0 END)
FROM workspace.scribd_exercise.exercise_content_items

UNION ALL

SELECT 'content_type',
SUM(CASE WHEN content_type IS NULL THEN 1 ELSE 0 END),
SUM(CASE WHEN TRIM(content_type) = '' THEN 1 ELSE 0 END),
SUM(CASE WHEN LOWER(TRIM(content_type)) IN ('n/a','na','null','none','unknown','-') THEN 1 ELSE 0 END)
FROM workspace.scribd_exercise.exercise_content_items

UNION ALL

SELECT 'publish_date',
SUM(CASE WHEN publish_date IS NULL THEN 1 ELSE 0 END), 0, 0
FROM workspace.scribd_exercise.exercise_content_items

UNION ALL

SELECT 'runtime_minutes_fake',
SUM(CASE WHEN runtime_minutes_fake IS NULL THEN 1 ELSE 0 END), 0, 0
FROM workspace.scribd_exercise.exercise_content_items;

column_name,nulls,blanks,placeholders
content_id,0,0,0
business_id,0,0,0
creator_id,195,0,0
title,0,0,0
category,0,0,0
content_type,0,0,0
publish_date,0,0,0
runtime_minutes_fake,0,0,0


**Finding:** Only creator_id has missing values: 195 of 300 (65%) are null. No blanks or placeholder text ("N/A", "unkown") in any column. Key colums (content_id, business_id, publish_date) are complete. 

In [0]:
-- Where the missing values are, by business 
SELECT ci.business_id, 
b.business_name, 
b.business_line, 
COUNT(*) AS content_items,
SUM(CASE WHEN ci.creator_id IS NULL THEN 1 ELSE 0 END) AS missing_creator
FROM workspace.scribd_exercise.exercise_content_items ci
JOIN workspace.scribd_exercise.exercise_businesses b
ON ci.business_id = b.business_id
GROUP BY ci.business_id, b.business_name, b.business_line 
ORDER BY ci.business_id;

business_id,business_name,business_line,content_items,missing_creator
BIZ_1,PageGarden Premium (SYNTHETIC),premium_published_content,112,112
BIZ_2,GatherSquare Community (SYNTHETIC),consumer_community,83,83
BIZ_3,MakerHarbor (SYNTHETIC),creator_content,105,0


%md
**Finding:** All 195 missing creator_id values are in BIZ_1 (premium_published_content, 112) and BIZ_2 (consumer_community, 83). BIZ_3 (creator_content) has no missing creator_id (0 of 105).

**Interpretation:** creator_id is "not applicable" outside the creator_content business line, not missing data.

**Treatment:** No action. Creator metrics will be scoped to BIZ_3 only.

### Relationships - Does every content item link to a valid business and a valid creator from the same business? 

In [0]:
-- Does every business id exists?
SELECT ci.business_id, 
COUNT(*) AS content_items 
FROM workspace.scribd_exercise.exercise_content_items ci 
LEFT ANTI JOIN workspace.scribd_exercise.exercise_businesses b
ON ci.business_id = b.business_id 
GROUP BY ci.business_id;

business_id,content_items


In [0]:
-- Does every creator id exists?
SELECT ci.creator_id
,COUNT(*) AS content_items 
FROM workspace.scribd_exercise.exercise_content_items ci 
LEFT ANTI JOIN workspace.scribd_exercise.exercise_creators c
ON ci.creator_id = c.creator_id 
WHERE ci.creator_id IS NOT NULL -- Excludes the 195 nulls from BIZ_1 AND BIZ_2
GROUP BY ci.creator_id;

creator_id,content_items


In [0]:
-- Do content and its creator belong to the same business? 
SELECT ci.business_id AS content_business,
cr.business_id AS creator_business,
COUNT(DISTINCT ci.content_id) AS content_items,
COUNT(DISTINCT cr.creator_id) AS creators
FROM workspace.scribd_exercise.exercise_content_items ci
LEFT JOIN workspace.scribd_exercise.exercise_creators cr
  ON ci.creator_id = cr.creator_id
GROUP BY ci.business_id, cr.business_id
ORDER BY ci.business_id, cr.business_id;

content_business,creator_business,content_items,creators
BIZ_1,null,112,0
BIZ_2,null,83,0
BIZ_3,BIZ_3,105,34


In [0]:
-- From the creators side, which creators have content
SELECT cr.business_id,
COUNT(DISTINCT cr.creator_id) AS creators,
COUNT(DISTINCT CASE WHEN ci.content_id IS NOT NULL THEN cr.creator_id END) AS creators_with_content,
COUNT(DISTINCT CASE WHEN ci.content_id IS NULL THEN cr.creator_id END) AS creators_without_content
FROM workspace.scribd_exercise.exercise_creators cr
LEFT JOIN workspace.scribd_exercise.exercise_content_items ci
  ON cr.creator_id = ci.creator_id
GROUP BY cr.business_id
ORDER BY cr.business_id;

business_id,creators,creators_with_content,creators_without_content
BIZ_1,3,0,3
BIZ_3,37,34,3


**Findings** 

1. Every content item links to an existing business, every non-null creator_id exists in the creators table, and all creator-linked content belongs to the same business as its creator (BIZ_3)
2. BIZ_3 has 37 creators, 34 with content; the 3 without content are treated as not yet published (normal). BIZ_1 has 3 creators, none with content, in a business with no creator content.

**Treatment:**
- None for content. BIZ_1 creators flagged and excluded from active-creator counts.

### Categories and content types - Do all category and content type values make sense? 

In [0]:
-- Content type counts 
SELECT content_type,
COUNT(*) AS items, 
ROUND(100.0 * COUNT(*) /  SUM(COUNT(*)) OVER(), 1) AS pct_of_content 
FROM workspace.scribd_exercise.exercise_content_items
GROUP BY content_type
ORDER BY items DESC;


content_type,items,pct_of_content
forum_post,53,17.7
serialized_chapter,52,17.3
audiobook,52,17.3
ebook,47,15.7
poll,46,15.3
video,42,14.0
TEST_TYPE,8,2.7


In [0]:
-- Category counts
SELECT category, 
COUNT(*) AS items, 
ROUND(100.0 * COUNT(*) /SUM(COUNT(*)) OVER(), 1) AS pct_of_content
FROM workspace.scribd_exercise.exercise_content_items
GROUP BY category
ORDER BY items DESC;

category,items,pct_of_content
Indie Fiction,29,9.7
Serialized Drama,26,8.7
Ask the Community,22,7.3
Fake History of Imaginary Countries,22,7.3
How-To Guides,21,7.0
Self-Help,20,6.7
Fantasy,19,6.3
Memes About Reading,17,5.7
General Nonsense,17,5.7
Poetry (Bad),16,5.3


In [0]:
-- Do types and categories fit their business? 
SELECT b.business_id
,b.business_line 
,ci.content_type
,ci.category 
,COUNT(*) AS items 
FROM workspace.scribd_exercise.exercise_content_items ci
LEFT JOIN workspace.scribd_exercise.exercise_businesses b
ON ci.business_id = b.business_id
GROUP BY b.business_id
,b.business_line 
,ci.content_type
,ci.category 
ORDER BY b.business_id, ci.content_type DESC;

business_id,business_line,content_type,category,items
BIZ_1,premium_published_content,video,Self-Help,3
BIZ_1,premium_published_content,video,Sci-Fi,3
BIZ_1,premium_published_content,video,Cooking,2
BIZ_1,premium_published_content,video,Fake History of Imaginary Countries,2
BIZ_1,premium_published_content,video,Fantasy,3
BIZ_1,premium_published_content,video,Mystery,3
BIZ_1,premium_published_content,serialized_chapter,Romance,2
BIZ_1,premium_published_content,serialized_chapter,Fake History of Imaginary Countries,4
BIZ_1,premium_published_content,serialized_chapter,Sci-Fi,1
BIZ_1,premium_published_content,serialized_chapter,Mystery,2


**Finding:** 
- 7 content types. 6 are valid formats; TEST_TYPE (8 items, 2.7%) marks test records.
- 17 categories, each with 10–29 items. No test values or rare one-offs.
- Checked visually (7 types, 17 categories); no spelling, spacing or capitalization variants.
-Categories are cleanly separated by business (BIZ_1: 7, BIZ_2: 5, BIZ_3: 5; no overlap). All 6 valid content types appear in all 3 businesses, so content_type is not business-specific and it will be used only to exclude test records.
- The 8 TEST_TYPE items are in BIZ_1 (5) and BIZ_2 (3); none in BIZ_3. They have normal categories, so only content_type identifies them.

**Treatment:** TEST_TYPE items will be quarantined from the clean layer; impact on engagement checked in Step 8.

**Impact of removing test items:** content count 292 instead of 300 (BIZ_1: 107, BIZ_2: 80, BIZ_3: 105).

### Titles - Does the same content appear under different IDs and are there test-lookin titles beyond TEST_TYPE? 

In [0]:
-- Same title under different IDs 
SELECT title,
COUNT(DISTINCT content_id) AS ids,
COLLECT_SET(content_id) AS content_ids,
COLLECT_SET(business_id) AS businesses,
COLLECT_SET(content_type) AS types
FROM workspace.scribd_exercise.exercise_content_items
GROUP BY title
HAVING COUNT(DISTINCT content_id) > 1
ORDER BY ids DESC;

title,ids,content_ids,businesses,types


In [0]:
-- How do test items look like
SELECT content_id, business_id, category, title, publish_date
FROM workspace.scribd_exercise.exercise_content_items
WHERE content_type = 'TEST_TYPE'
ORDER BY business_id, content_id;

content_id,business_id,category,title,publish_date
CNT_00001,BIZ_1,Sci-Fi,The Great Prairiefox (Mock Book #1),2023-11-02
CNT_00066,BIZ_1,Romance,The Last Recipe (Mock Book #66),2025-10-10
CNT_00162,BIZ_1,Self-Help,The Great Algorithm (Mock Book #162),2023-03-11
CNT_00223,BIZ_1,Fake History of Imaginary Countries,The Last Recipe (Mock Book #223),2025-09-10
CNT_00272,BIZ_1,Fake History of Imaginary Countries,The Last Prairiefox (Mock Book #272),2025-02-08
CNT_00002,BIZ_2,Book Club Chatter,[MOCK THREAD] Rant: fake topic #2,2021-08-01
CNT_00130,BIZ_2,Memes About Reading,[MOCK THREAD] Unpopular opinion: fake topic #130,2023-02-16
CNT_00256,BIZ_2,General Nonsense,[MOCK THREAD] Unpopular opinion: fake topic #256,2022-09-02


**Finding:** 
- No duplicate titles across content IDs
- The 8 TEST_TYPE items look like real content (normal titles and categories) and were published across 2021–2025, not as one test batch. Only content_type identifies them.

**Treatment:** No new issue. Confirms the quarantine approach for issue #4.

### Publish dates - Are publish dates within a normal range for each business? Does engagement only happen after publishing?

In [0]:
-- Date range per business, compared to it's launch
SELECT b.business_id 
, b.business_name
, b.business_line
, b.launch_date
, MIN(ci.publish_date) AS min_date
, MAX(ci.publish_date) AS max_date
, DATEDIFF(B.launch_date, MIN(ci.publish_date)) AS max_days_pre_launch
, SUM(CASE WHEN ci.publish_date < b.launch_date THEN 1 ELSE 0 END) AS pre_launch
, SUM(CASE WHEN ci.publish_date < b.launch_date AND ci.content_type == 'TEST_TYPE' THEN 1 ELSE 0 END) as pre_launch_test_content_type
FROM workspace.scribd_exercise.exercise_businesses b
LEFT JOIN workspace.scribd_exercise.exercise_content_items ci
ON b.business_id = ci.business_id
GROUP BY b.business_id
, b.business_name
, b.business_line
, b.launch_date
ORDER BY b.business_id;

business_id,business_name,business_line,launch_date,min_date,max_date,max_days_pre_launch,pre_launch,pre_launch_test_content_type
BIZ_1,PageGarden Premium (SYNTHETIC),premium_published_content,2021-01-01,2020-12-24,2026-09-09,8,2,0
BIZ_2,GatherSquare Community (SYNTHETIC),consumer_community,2021-06-15,2021-07-14,2026-08-02,-29,0,0
BIZ_3,MakerHarbor (SYNTHETIC),creator_content,2022-03-01,2022-01-22,2026-09-14,38,1,0


In [0]:
-- Content published per year
SELECT DATE_TRUNC('year', publish_date) AS year, 
business_id, 
COUNT(*) AS items
FROM workspace.scribd_exercise.exercise_content_items
GROUP BY DATE_TRUNC('year', publish_date), business_id
ORDER BY year, business_id;

year,business_id,items
2020-01-01T00:00:00.000Z,BIZ_1,2
2021-01-01T00:00:00.000Z,BIZ_1,25
2021-01-01T00:00:00.000Z,BIZ_2,6
2022-01-01T00:00:00.000Z,BIZ_1,16
2022-01-01T00:00:00.000Z,BIZ_2,23
2022-01-01T00:00:00.000Z,BIZ_3,11
2023-01-01T00:00:00.000Z,BIZ_1,17
2023-01-01T00:00:00.000Z,BIZ_2,17
2023-01-01T00:00:00.000Z,BIZ_3,10
2024-01-01T00:00:00.000Z,BIZ_1,14


Databricks visualization. Run in Databricks to view.

In [0]:
-- Engagement before the content was published 
SELECT ci.business_id,
COUNT(*) AS event_rows_before_publish,
COUNT(DISTINCT ci.content_id) AS content_items_affected,
MIN(DATEDIFF(ci.publish_date, CAST(e.event_timestamp AS DATE))) AS min_days_early,
MAX(DATEDIFF(ci.publish_date, CAST(e.event_timestamp AS DATE))) AS max_days_early
FROM workspace.scribd_exercise.exercise_engagement_events e
JOIN workspace.scribd_exercise.exercise_content_items ci
ON e.content_id = ci.content_id
WHERE CAST(e.event_timestamp AS DATE) < ci.publish_date
GROUP BY ci.business_id
ORDER BY ci.business_id;

business_id,event_rows_before_publish,content_items_affected,min_days_early,max_days_early
BIZ_1,10,9,6,1516
BIZ_2,4,4,24,628
BIZ_3,19,18,4,1880


In [0]:
SELECT ci.business_id,
       CASE WHEN DATEDIFF(ci.publish_date, CAST(e.event_timestamp AS DATE)) <= 30
            THEN '1–30 days early' ELSE 'more than 30 days early' END AS gap,
       COUNT(*) AS event_rows,
       SUM(CASE WHEN CAST(e.event_timestamp AS DATE) < b.launch_date THEN 1 ELSE 0 END) AS before_business_launch,
       SUM(CASE WHEN ci.content_type = 'TEST_TYPE' THEN 1 ELSE 0 END) AS on_test_content
FROM workspace.scribd_exercise.exercise_engagement_events e
JOIN workspace.scribd_exercise.exercise_content_items ci ON e.content_id = ci.content_id
JOIN workspace.scribd_exercise.exercise_businesses b ON ci.business_id = b.business_id
WHERE CAST(e.event_timestamp AS DATE) < ci.publish_date
GROUP BY 1, 2
ORDER BY 1, 2;

business_id,gap,event_rows,before_business_launch,on_test_content
BIZ_1,1–30 days early,7,0,0
BIZ_1,more than 30 days early,3,0,0
BIZ_2,1–30 days early,2,0,0
BIZ_2,more than 30 days early,2,0,0
BIZ_3,1–30 days early,8,0,0
BIZ_3,more than 30 days early,11,3,0


**Findings:** 
- 3 of 300 items (1%) were published before their own business launched: 2 in BIZ_1 (up to 8 days early) and 1 in BIZ_3 (38 days early). None are test items. BIZ_2's first content is 29 days after its launch.
- Counts add up to 300 (BIZ_1 112 + BIZ_2 83 + BIZ_3 105), matching earlier steps. Empty months occur throughout the history, consistent with low volume (0–5 items per business per month), not a pipeline break.
- **BIZ_3 (creator_content) is growing fast:** 11 → 10 → 16 → 27 → 41 items per year; 2026 is already the highest year despite covering only about 8.5 months. Its busiest months (10 in Feb 2026, 9 in Jul 2026) fit this growth and are not errors.
- **BIZ_1 and BIZ_2 are flat or uneven** year to year, with no clear trend.
- Of the 33 events dated before publish, 17 are 1–30 days early and 16 are more than 30 days early, including 3 BIZ_3 events dated before the business launched. None are on test content.

**Interpretation:** consistent with pre-launch catalog loading or a soft launch.

**Treatment:** 
- Keep items published before business launch; flag as "published before launch." Low impact (only affects pre-launch months).
- **Don't report content published per month by business as a KPI.** At 0–5 items a month, changes are mostly noise (2 → 5 items looks like +150%). Quarterly or yearly views are better.
- **Highlight BIZ_3 growth:** the creator business is the fastest-growing content source. This describes a trend in volume, not its cause.
- **Compare like with like:** 2026 only runs to mid-September, so compare the same months across years (e.g., January–August).
- Quarantine the 16 engagement events (> 30 days early); keep and flag the 17 engagemment events (1–30 days early).
 

### Runtime - Does runtime_minutes_fake contain real content lenghts, or placeholder values that make it useless?

In [0]:
-- Runtime value counts
SELECT runtime_minutes_fake
, COUNT(*) AS items
,ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER(), 1) AS pct_of_content
FROM workspace.scribd_exercise.exercise_content_items
GROUP BY runtime_minutes_fake
ORDER BY runtime_minutes_fake DESC;

runtime_minutes_fake,items,pct_of_content
99999,5,1.7
180,18,6.0
90,31,10.3
45,52,17.3
25,53,17.7
12,42,14.0
8,48,16.0
5,30,10.0
2,16,5.3
-1,5,1.7


**Findings:** 
- 10 of 300 items (3.3%) have placeholder runtimes: 5 at -1 and 5 at 99999. The other 290 (96.7%) take one of 8 values: 2, 5, 8, 12, 25, 45, 90 or 180 minutes.

**Interpretation:** the column is mostly usable, but the values look like preset length bands rather than real measured runtimes.

**Treatment:** convert placeholders to null and flag them (runtime_placeholder); use runtime for length groupings, not precise averages.

In [0]:
-- Runtime by content type 
SELECT content_type
, runtime_minutes_fake
,COUNT(*) AS items 
FROM workspace.scribd_exercise.exercise_content_items
GROUP BY content_type, runtime_minutes_fake
ORDER BY content_type, runtime_minutes_fake;

content_type,runtime_minutes_fake,items
TEST_TYPE,5,3
TEST_TYPE,12,5
audiobook,45,17
audiobook,90,16
audiobook,180,18
audiobook,99999,1
ebook,25,16
ebook,45,16
ebook,90,15
forum_post,-1,1


**Findings** 
- Runtime bands differ by format in a sensible way (poll 2–8 min; forum_post 5–12; video 8–45; serialized_chapter 12–45; ebook 25–90; audiobook 45–180), suggesting estimated time to consume rather than measured length. 
-The 10 placeholders (-1 and 99999) are spread across formats that have lengths, so they don't mean "not applicable." 
-TEST_TYPE items have normal-looking runtimes. Another sign that only content_type identifies them. 

**Treatment:** keep rows; set runtime to null and flag runtime_placeholder (10 items). Use runtime as a length band within each format, not as a precise average.

### Engagement - Do test items have engagement events? Who generated them? Does any content show outlier engagement? 

In [0]:
-- Do test items have events 
 SELECT ci.business_id,
COUNT(DISTINCT ci.content_id) AS test_items,
COUNT(DISTINCT CASE WHEN e.event_id IS NOT NULL THEN ci.content_id END) AS test_items_with_events,
COUNT(e.event_id) AS event_rows,
COUNT(DISTINCT e.event_id) AS distinct_events,
COUNT(DISTINCT e.user_id) AS users
FROM workspace.scribd_exercise.exercise_content_items ci
LEFT JOIN workspace.scribd_exercise.exercise_engagement_events e
ON ci.content_id = e.content_id
WHERE ci.content_type = 'TEST_TYPE'
GROUP BY ci.business_id
ORDER BY ci.business_id;

business_id,test_items,test_items_with_events,event_rows,distinct_events,users
BIZ_1,5,5,104,104,98
BIZ_2,3,3,63,63,61


**Findings:** 
- All 8 TEST_TYPE items have engagement: 167 events (BIZ_1: 104, BIZ_2: 63) from 159 users, with no duplicate events. That's about 20 events per item, similar to the average for all content, suggesting test content was engaged with like real content.

In [0]:
-- How big is that compared with all engagement? 
SELECT ci.business_id,
       COUNT(*) AS event_rows_with_content,
       SUM(CASE WHEN ci.content_type = 'TEST_TYPE' THEN 1 ELSE 0 END) AS event_rows_on_test_content,
       ROUND(100.0 * SUM(CASE WHEN ci.content_type = 'TEST_TYPE' THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_on_test_content
FROM workspace.scribd_exercise.exercise_engagement_events e
JOIN workspace.scribd_exercise.exercise_content_items ci
  ON e.content_id = ci.content_id
GROUP BY ci.business_id
ORDER BY ci.business_id;

business_id,event_rows_with_content,event_rows_on_test_content,pct_on_test_content
BIZ_1,2078,104,5.00
BIZ_2,1627,63,3.87
BIZ_3,1972,0,0.00


**Finding:** 
- Test content accounts for 167 of 5,677 events with content (2.9%): 5.0% of BIZ_1's engagement (104 of 2,078) and 3.9% of BIZ_2's (63 of 1,627). BIZ_3 has none.

**Impact:** without cleaning, BIZ_1 and BIZ_2 engagement would be overstated, and comparisons with BIZ_3 skewed.

**Treatment:** quarantine the 167 events on test content (reason: test_content) in the events cleaning.

In [0]:
-- Who generated the events on test content? 
SELECT CASE
         WHEN u.user_id IS NULL THEN 'user not in users table'
         WHEN u.is_test = 1 THEN 'test account'
         ELSE 'regular account'
       END AS account_type,
       COUNT(*) AS event_rows,
       COUNT(DISTINCT e.user_id) AS users
FROM workspace.scribd_exercise.exercise_engagement_events e
JOIN workspace.scribd_exercise.exercise_content_items ci
  ON e.content_id = ci.content_id
LEFT JOIN (
  SELECT user_id, MAX(CAST(is_test_account_flag AS INT)) AS is_test
  FROM workspace.scribd_exercise.exercise_users
  GROUP BY user_id
) u ON e.user_id = u.user_id
WHERE ci.content_type = 'TEST_TYPE'
GROUP BY 1
ORDER BY event_rows DESC;

account_type,event_rows,users
regular account,161,147
test account,6,6


**Findings:** 
- 161 of the 167 events on test content (96%) came from 147 regular users; only 6 came from test accounts. 
- Test content was visible to and used by real customers.

**Treatment:** quarantine the 6 test-account events; keep the 161 regular-user events with dq_flag = on_test_content (excluded from content engagement metrics, included in MAU).

In [0]:
-- Engagement per content item (outliers and zero-engagement content)
WITH per_item AS (
  SELECT ci.content_id, ci.business_id, ci.content_type,
         COUNT(e.event_id) AS events
  FROM workspace.scribd_exercise.exercise_content_items ci
  LEFT JOIN workspace.scribd_exercise.exercise_engagement_events e
    ON ci.content_id = e.content_id
  GROUP BY ci.content_id, ci.business_id, ci.content_type
)
SELECT business_id,
       COUNT(*) AS items,
       SUM(CASE WHEN events = 0 THEN 1 ELSE 0 END) AS items_with_zero_events,
       MIN(events) AS min_events,
       PERCENTILE(events, 0.5) AS median_events,
       ROUND(AVG(events), 1) AS avg_events,
       MAX(events) AS max_events
FROM per_item
GROUP BY business_id
ORDER BY business_id;

business_id,items,items_with_zero_events,min_events,median_events,avg_events,max_events
BIZ_1,112,0,10,18.0,18.6,31
BIZ_2,83,0,11,20.0,19.6,29
BIZ_3,105,0,9,19.0,18.8,27


**Finding:** 
- No engagement outliers and no content without engagement: 9 to 31 events per item, median 18–20 in every business.

**Observation:** engagement is unusually even across content (real engagement is usually highly skewed), so content popularity rankings would mostly reflect noise.

# Creators Table

In [0]:
DESCRIBE TABLE workspace.scribd_exercise.exercise_creators;

col_name,data_type,comment
creator_id,string,null
creator_name,string,null
business_id,string,null
join_date,date,null
payout_tier,string,null
follower_count_fake,bigint,null
# Clustering Information,,
# col_name,data_type,comment


In [0]:
SELECT * FROM workspace.scribd_exercise.exercise_creators LIMIT 10;

creator_id,creator_name,business_id,join_date,payout_tier,follower_count_fake
CR_0001,Fable Sampleridge (Creator),BIZ_3,2022-02-05,Tier_1_Standard,-5
CR_0002,Denny Sampleridge (Creator),BIZ_1,2020-12-07,Tier_3_Partner,-5
CR_0003,Tessa Fableton (Creator),BIZ_3,2023-06-15,Tier_1_Standard,999999
CR_0004,Tessa Cottontail (Creator),BIZ_3,2025-09-15,Tier_3_Partner,999999
CR_0005,Finley Trialton (Creator),BIZ_3,2022-10-16,Tier_2_Growth,1475
CR_0006,Denny Cottontail (Creator),BIZ_3,2024-10-10,Tier_0_Starter,4232
CR_0007,Moxie Beetlebrook (Creator),BIZ_3,2024-11-09,Tier_2_Growth,3538
CR_0008,Quinn Sampleridge (Creator),BIZ_3,2025-03-31,Tier_1_Standard,12662
CR_0009,Fable Beetlebrook (Creator),BIZ_3,2023-02-27,Tier_0_Starter,2375
CR_0010,Buzzy Storyfield (Creator),BIZ_3,2022-04-07,Tier_0_Starter,11845


### Uniqueness - Is each creator listed once, and if not, do the copies agree?

In [0]:
-- Exact counts 
SELECT COUNT(*) AS total_rows
,COUNT (DISTINCT creator_id) AS distinct_ids 
,COUNT(*) - COUNT(DISTINCT creator_id) AS extra_rows
FROM workspace.scribd_exercise.exercise_creators;

total_rows,distinct_ids,extra_rows
40,40,0


**Findings:** 
- No duplicate creators: 40 rows, 40 unique creator IDs. No creator is registered under more than one business.

### Completeness - Are any values missing in the creators table?

In [0]:
-- AI generated 
SELECT 'creator_id' AS column_name,
       SUM(CASE WHEN creator_id IS NULL THEN 1 ELSE 0 END) AS nulls,
       SUM(CASE WHEN TRIM(creator_id) = '' THEN 1 ELSE 0 END) AS blanks,
       SUM(CASE WHEN LOWER(TRIM(creator_id)) IN ('n/a','na','null','none','unknown','-') THEN 1 ELSE 0 END) AS placeholders
FROM workspace.scribd_exercise.exercise_creators
UNION ALL
SELECT 'creator_name',
       SUM(CASE WHEN creator_name IS NULL THEN 1 ELSE 0 END),
       SUM(CASE WHEN TRIM(creator_name) = '' THEN 1 ELSE 0 END),
       SUM(CASE WHEN LOWER(TRIM(creator_name)) IN ('n/a','na','null','none','unknown','-') THEN 1 ELSE 0 END)
FROM workspace.scribd_exercise.exercise_creators
UNION ALL
SELECT 'business_id',
       SUM(CASE WHEN business_id IS NULL THEN 1 ELSE 0 END),
       SUM(CASE WHEN TRIM(business_id) = '' THEN 1 ELSE 0 END),
       SUM(CASE WHEN LOWER(TRIM(business_id)) IN ('n/a','na','null','none','unknown','-') THEN 1 ELSE 0 END)
FROM workspace.scribd_exercise.exercise_creators
UNION ALL
SELECT 'payout_tier',
       SUM(CASE WHEN payout_tier IS NULL THEN 1 ELSE 0 END),
       SUM(CASE WHEN TRIM(payout_tier) = '' THEN 1 ELSE 0 END),
       SUM(CASE WHEN LOWER(TRIM(payout_tier)) IN ('n/a','na','null','none','unknown','-') THEN 1 ELSE 0 END)
FROM workspace.scribd_exercise.exercise_creators
UNION ALL
SELECT 'join_date',
       SUM(CASE WHEN join_date IS NULL THEN 1 ELSE 0 END), 0, 0
FROM workspace.scribd_exercise.exercise_creators
UNION ALL
SELECT 'follower_count_fake',
       SUM(CASE WHEN follower_count_fake IS NULL THEN 1 ELSE 0 END), 0, 0
FROM workspace.scribd_exercise.exercise_creators;

column_name,nulls,blanks,placeholders
creator_id,0,0,0
creator_name,0,0,0
business_id,0,0,0
payout_tier,0,0,0
join_date,0,0,0
follower_count_fake,0,0,0


**Findings:** 
- No missing values (nulls, blanks or placeholder text) in any column.

### Valid values - Are payout tiers valid and consistent? Are follower counts plausible? 

In [0]:
-- Payout tiers counts by business
SELECT business_id 
, payout_tier 
, COUNT (*) AS creators
FROM workspace.scribd_exercise.exercise_creators
GROUP BY 1,2
ORDER BY 1,2;

business_id,payout_tier,creators
BIZ_1,Tier_1_Standard,1
BIZ_1,Tier_3_Partner,2
BIZ_3,Tier_0_Starter,9
BIZ_3,Tier_1_Standard,17
BIZ_3,Tier_2_Growth,7
BIZ_3,Tier_3_Partner,4


**Findings:** 
- 4 valid payout tiers (Tier_0_Starter to Tier_3_Partner), consistently named. 
- BIZ_3 has creators in all 4 tiers
- BIZ_2 has no creators
- BIZ_1's 3 creators (with no content) include 2 at Tier_3_Partner, the top tier.

**Interpretation:** top-tier partners with no content in a business without creator content strengthens the case that the BIZ_1 records are misassigned or invalid (but it would require some confirmation). They stay excluded from creator metrics.

In [0]:
-- Follower count distribution 
SELECT COUNT(*) AS creators,
SUM(CASE WHEN follower_count_fake = 999999 THEN 1 ELSE 0 END) AS placeholder_999999,
SUM(CASE WHEN follower_count_fake < 0 THEN 1 ELSE 0 END) AS negative,
SUM(CASE WHEN follower_count_fake = 0 THEN 1 ELSE 0 END) AS zero,
MIN(CASE WHEN follower_count_fake <> 999999 AND follower_count_fake >= 0 THEN follower_count_fake END) AS min_followers,
PERCENTILE(CASE WHEN follower_count_fake <> 999999 AND follower_count_fake >= 0 THEN follower_count_fake END, 0.25) AS p25,
PERCENTILE(CASE WHEN follower_count_fake <> 999999 AND follower_count_fake >= 0 THEN follower_count_fake END, 0.5) AS median,
PERCENTILE(CASE WHEN follower_count_fake <> 999999 AND follower_count_fake >= 0 THEN follower_count_fake END, 0.75) AS p75,
MAX(CASE WHEN follower_count_fake <> 999999 AND follower_count_fake >= 0 THEN follower_count_fake END) AS max_followers
FROM workspace.scribd_exercise.exercise_creators;

creators,placeholder_999999,negative,zero,min_followers,p25,median,p75,max_followers
40,2,2,0,177,1844.75,9053.0,27653.5,158177


**Findings:** 
- 4 of 40 creators (10%) have placeholder follower counts: 2 negative and 2 at 999999. Excluding them, follower counts range from 177 to 158,177 (median 9,053), a plausible long-tailed distribution. 

**Treatment:** keep rows; set follower count to null and flag follower_placeholder.

In [0]:
-- Detailed extreme values 
SELECT creator_id, business_id, payout_tier, join_date, follower_count_fake
FROM workspace.scribd_exercise.exercise_creators
ORDER BY follower_count_fake DESC
LIMIT 5;

creator_id,business_id,payout_tier,join_date,follower_count_fake
CR_0004,BIZ_3,Tier_3_Partner,2025-09-15,999999
CR_0003,BIZ_3,Tier_1_Standard,2023-06-15,999999
CR_0015,BIZ_3,Tier_1_Standard,2024-01-02,158177
CR_0027,BIZ_3,Tier_1_Standard,2022-04-23,151278
CR_0014,BIZ_3,Tier_1_Standard,2024-08-21,121503


**Findings:** 
- **Lowest:** The 2 negative-follower creators are CR_0001 (BIZ_3) and CR_0002 (BIZ_1), the same 2 creators who joined before their business launched, and the first 2 IDs in the table. CR_0002 also has no content, is top tier, and sits in a business without creator content. Multiple anomalies on the earliest records suggest possible seed or test records.
- **Highest:** The 2 creators at 999999 are CR_0003 and CR_0004 (both BIZ_3). All 4 placeholder creators are CR_0001–CR_0004, the first 4 IDs. Across tables, anomalies cluster in the earliest records (test content includes CNT_00001 and CNT_00002), suggesting possible seed or test records.
- The highest real follower counts (121k–158k) are all Tier_1_Standard. This suggests that payout tiers may depend on something other than jsut followers. 

### Join dates - Are join dates plausible? Does every creator join before publishing content?

In [0]:
-- Creators who published content before they joined 
SELECT cr.creator_id
, cr.business_id 
, cr.join_date
, MIN(ct.publish_date) AS first_publish_date
, DATEDIFF(cr.join_date, MIN(ct.publish_date)) AS days_content_before_join
FROM workspace.scribd_exercise.exercise_creators cr
JOIN workspace.scribd_exercise.exercise_content_items ct
ON cr.creator_id = ct.creator_id
GROUP BY 1,2,3
HAVING MIN(ct.publish_date) < cr.join_date
ORDER BY days_content_before_join DESC;


creator_id,business_id,join_date,first_publish_date,days_content_before_join
CR_0023,BIZ_3,2022-06-09,2022-01-22,138
CR_0013,BIZ_3,2026-03-09,2026-02-05,32
CR_0031,BIZ_3,2022-05-11,2022-04-13,28
CR_0010,BIZ_3,2022-04-07,2022-03-20,18
CR_0022,BIZ_3,2024-10-25,2024-10-12,13


**Findings:** 
- 5 of 34 active BIZ_3 creators (15%) published content before their join date, by 13 to 138 days. 3 of the 5 are around BIZ_3's launch in 2022. CR_0023's first content (2022-01-22) is also the BIZ_3 item published before the business launched, suggesting a wrong publish date for that item.

**Treatment:** keep; flag content_before_join. Open question: does join_date mean account creation or a later step, such as joining the creator program?

In [0]:
-- New creators per year 
SELECT YEAR(join_date) AS join_year
, business_id
, COUNT(*) AS new_creators
FROM workspace.scribd_exercise.exercise_creators 
GROUP BY 1,2 
ORDER BY 2,1;

join_year,business_id,new_creators
2020,BIZ_1,1
2021,BIZ_1,1
2023,BIZ_1,1
2022,BIZ_3,9
2023,BIZ_3,8
2024,BIZ_3,8
2025,BIZ_3,9
2026,BIZ_3,3


**Findings:** 
- New creators per year reconcile to totals (BIZ_1: 3; BIZ_3: 37). 
- BIZ_3 signups are flat (8–9 per year in 2022–2025; 3 in 2026 through August) 
- BIZ_3 content grew sharply (27 items in 2025, 41 in 2026), so content growth appears to come from existing creators publishing more, not from more creators.

# Users Table

In [0]:
DESCRIBE TABLE workspace.scribd_exercise.exercise_users;

col_name,data_type,comment
user_id,string,null
user_name,string,null
email,string,null
signup_date,date,null
country,string,null
age_band,string,null
acquisition_channel,string,null
primary_device,string,null
is_test_account_flag,boolean,null
# Clustering Information,,


In [0]:
SELECT * FROM workspace.scribd_exercise.exercise_users LIMIT 10;

user_id,user_name,email,signup_date,country,age_band,acquisition_channel,primary_device,is_test_account_flag
USR_00001,Buzzy Prairiefox,usr00001@example.test,2021-03-02,Australia,18-24,influencer_TEST_CAMPAIGN,iOS_app,false
USR_00002,Tessa Skyward,usr00002@example.test,2022-11-06,Germany,45-54,organic_search,iOS_app,false
USR_00003,Bowie Cottontail,usr00003@example.test,2022-09-27,Japan,13-17,organic_search,Android_app,false
USR_00004,Zephyr Prairiefox,usr00004@example.test,2026-05-07,Brazil,18-24,referral,Android_app,false
USR_00005,Moxie Inkwell,usr00005@example.test,2023-08-22,Altara,N/A,paid_social,web_mobile,false
USR_00006,Finley Skyward,usr00006@example.test,2024-07-10,Latoria,45-54,paid_social,web_mobile,false
USR_00007,Rook Sampleridge,usr00007@example.test,2026-02-21,Japan,35-44,paid_social,Android_app,false
USR_00008,Izzy Storyfield,usr00008@example.test,2021-10-09,Germany,13-17,referral,web_mobile,false
USR_00009,Denny Holdsworth,usr00009@example.test,2024-05-11,Hogglewick,25-34,referral,web_desktop,false
USR_00010,Wiley Holdsworth,usr00010@example.test,2023-05-13,Genora,13-17,carrier_pigeon,web_mobile,false


### Uniqueness - is each user listed once?

In [0]:
SELECT COUNT(*) AS total_rows,
COUNT(DISTINCT user_id) AS distinct_ids,
COUNT(*) - COUNT(DISTINCT user_id) AS extra_rows,
COUNT(DISTINCT email) AS distinct_emails
FROM workspace.scribd_exercise.exercise_users;

total_rows,distinct_ids,extra_rows,distinct_emails
806,800,6,800


In [0]:
--Duplicated IDs: exact copies or not? 
SELECT COUNT(*) AS duplicated_ids,
SUM(copies) AS rows_involved,
SUM(CASE WHEN distinct_versions = 1 THEN 1 ELSE 0 END) AS ids_exact_copies,
SUM(CASE WHEN distinct_versions > 1 THEN 1 ELSE 0 END) AS ids_conflicting
FROM (
  SELECT user_id,
    COUNT(*) AS copies,
    COUNT(DISTINCT TO_JSON(STRUCT(*))) AS distinct_versions --Shorcut: turns each whole row into one text value, so counting distinct versions tells you whether the copies are identical in every column.
  FROM workspace.scribd_exercise.exercise_users
  GROUP BY user_id
  HAVING COUNT(*) > 1
);

duplicated_ids,rows_involved,ids_exact_copies,ids_conflicting
6,12,6,0


In [0]:
-- Duplicated rows 
SELECT *
FROM workspace.scribd_exercise.exercise_users
WHERE user_id IN (
  SELECT user_id
  FROM workspace.scribd_exercise.exercise_users
  GROUP BY user_id
  HAVING COUNT(*) > 1)
ORDER BY user_id;

user_id,user_name,email,signup_date,country,age_band,acquisition_channel,primary_device,is_test_account_flag
USR_00064,Wiley Beetlebrook,usr00064@example.test,2025-08-20,Germany,13-17,organic_search,web_desktop,false
USR_00064,Wiley Beetlebrook,usr00064@example.test,2025-08-20,Germany,13-17,organic_search,web_desktop,false
USR_00067,Juno Noll,usr00067@example.test,2026-09-16,Freelandia,55-64,referral,web_mobile,false
USR_00067,Juno Noll,usr00067@example.test,2026-09-16,Freelandia,55-64,referral,web_mobile,false
USR_00534,Tully Holdsworth,usr00534@example.test,2023-04-11,Germany,25-34,organic_search,web_desktop,false
USR_00534,Tully Holdsworth,usr00534@example.test,2023-04-11,Germany,25-34,organic_search,web_desktop,false
USR_00622,Rook Noll,usr00622@example.test,2026-06-27,Genora,13-17,paid_social,Android_app,false
USR_00622,Rook Noll,usr00622@example.test,2026-06-27,Genora,13-17,paid_social,Android_app,false
USR_00668,Emery Beetlebrook,usr00668@example.test,2025-12-06,United Kingdom,35-44,email_campaign,web_mobile,false
USR_00668,Emery Beetlebrook,usr00668@example.test,2025-12-06,United Kingdom,35-44,email_campaign,web_mobile,false


**Findings** 
- 806 rows, 800 unique users. 
- 6 user IDs appear twice, all exact copies. 
- 800 unique emails, so no email is shared across different user IDs.

**Treatment:** keep one row per user; quarantine the 6 extra copies (duplicate_row). User count = 800, not 806.

**Why it matters:** joining events or subscriptions to users would double these users' activity.

### Completeness - Which colums have missing values? 

In [0]:
-- AI generated 
-- Missing values in every column 
SELECT 'user_id' AS column_name,
       SUM(CASE WHEN user_id IS NULL THEN 1 ELSE 0 END) AS nulls,
       SUM(CASE WHEN TRIM(user_id) = '' THEN 1 ELSE 0 END) AS blanks,
       SUM(CASE WHEN LOWER(TRIM(user_id)) IN ('n/a','na','null','none','unknown','-') THEN 1 ELSE 0 END) AS placeholders
FROM workspace.scribd_exercise.exercise_users
UNION ALL
SELECT 'user_name',
       SUM(CASE WHEN user_name IS NULL THEN 1 ELSE 0 END),
       SUM(CASE WHEN TRIM(user_name) = '' THEN 1 ELSE 0 END),
       SUM(CASE WHEN LOWER(TRIM(user_name)) IN ('n/a','na','null','none','unknown','-') THEN 1 ELSE 0 END)
FROM workspace.scribd_exercise.exercise_users
UNION ALL
SELECT 'email',
       SUM(CASE WHEN email IS NULL THEN 1 ELSE 0 END),
       SUM(CASE WHEN TRIM(email) = '' THEN 1 ELSE 0 END),
       SUM(CASE WHEN LOWER(TRIM(email)) IN ('n/a','na','null','none','unknown','-') THEN 1 ELSE 0 END)
FROM workspace.scribd_exercise.exercise_users
UNION ALL
SELECT 'country',
       SUM(CASE WHEN country IS NULL THEN 1 ELSE 0 END),
       SUM(CASE WHEN TRIM(country) = '' THEN 1 ELSE 0 END),
       SUM(CASE WHEN LOWER(TRIM(country)) IN ('n/a','na','null','none','unknown','-') THEN 1 ELSE 0 END)
FROM workspace.scribd_exercise.exercise_users
UNION ALL
SELECT 'primary_device',
       SUM(CASE WHEN primary_device IS NULL THEN 1 ELSE 0 END),
       SUM(CASE WHEN TRIM(primary_device) = '' THEN 1 ELSE 0 END),
       SUM(CASE WHEN LOWER(TRIM(primary_device)) IN ('n/a','na','null','none','unknown','-') THEN 1 ELSE 0 END)
FROM workspace.scribd_exercise.exercise_users
UNION ALL
SELECT 'age_band',
       SUM(CASE WHEN age_band IS NULL THEN 1 ELSE 0 END),
       SUM(CASE WHEN TRIM(age_band) = '' THEN 1 ELSE 0 END),
       SUM(CASE WHEN LOWER(TRIM(age_band)) IN ('n/a','na','null','none','unknown','-') THEN 1 ELSE 0 END)
FROM workspace.scribd_exercise.exercise_users
UNION ALL
SELECT 'acquisition_channel',
       SUM(CASE WHEN acquisition_channel IS NULL THEN 1 ELSE 0 END),
       SUM(CASE WHEN TRIM(acquisition_channel) = '' THEN 1 ELSE 0 END),
       SUM(CASE WHEN LOWER(TRIM(acquisition_channel)) IN ('n/a','na','null','none','unknown','-') THEN 1 ELSE 0 END)
FROM workspace.scribd_exercise.exercise_users
UNION ALL
SELECT 'signup_date',
       SUM(CASE WHEN signup_date IS NULL THEN 1 ELSE 0 END), 0, 0
FROM workspace.scribd_exercise.exercise_users
UNION ALL
SELECT 'is_test_account_flag',
       SUM(CASE WHEN is_test_account_flag IS NULL THEN 1 ELSE 0 END), 0, 0
FROM workspace.scribd_exercise.exercise_users;

column_name,nulls,blanks,placeholders
user_id,0,0,0
user_name,0,0,0
email,0,0,0
country,0,0,0
primary_device,0,0,0
age_band,20,0,20
acquisition_channel,20,0,0
signup_date,0,0,0
is_test_account_flag,0,0,0


**Findings:** 
- Only age_band and acquisition_channel have missing values.
- age_band is missing in 40 rows, recorded two ways (20 nulls, 20 'N/A') 
- acquisition_channel is null in 20 rows. 

In [0]:
-- Are the missing values in the same users or in test accounts? 
SELECT 
CASE WHEN age_band IS NULL OR TRIM(age_band) = '' OR LOWER(TRIM(age_band)) IN ('n/a','na','null','none','unknown','-')
THEN 'missing' ELSE 'present' END AS age_band_status,

CASE WHEN acquisition_channel IS NULL OR TRIM(acquisition_channel) = '' OR LOWER(TRIM(acquisition_channel)) IN ('n/a','na','null','none','unknown','-')
THEN 'missing' ELSE 'present' END AS channel_status,

is_test_account_flag,
COUNT(DISTINCT user_id) AS users
FROM workspace.scribd_exercise.exercise_users
GROUP BY 1, 2, 3
ORDER BY 1, 2, 3;

age_band_status,channel_status,is_test_account_flag,users
missing,missing,false,3
missing,present,false,36
missing,present,true,1
present,missing,false,15
present,missing,true,2
present,present,false,706
present,present,true,37


**Findings (2b):** 
- age_band is missing for 40 users (5.0%) and acquisition_channel for 20 (2.5%); 
- only 3 users are missing both, so the gaps are mostly independent.
- Most are regular accounts (39 of 40 and 18 of 20), so removing test accounts won't resolve them.

**Treatment:** keep users; standardize null and 'N/A' to 'Unknown' and show it as an explicit group in breakdowns (issue #12).

### Valid values - are the values in each column valid? 

In [0]:
-- Values in each column counted by unique user
WITH u AS (
  SELECT DISTINCT *
  FROM workspace.scribd_exercise.exercise_users
)
SELECT 'age_band' AS column_name, COALESCE(age_band, 'NULL') AS value, COUNT(*) AS users FROM u GROUP BY 2
UNION ALL
SELECT 'acquisition_channel', COALESCE(acquisition_channel, 'NULL'), COUNT(*) FROM u GROUP BY 2
UNION ALL
SELECT 'country', country, COUNT(*) FROM u GROUP BY 2
UNION ALL
SELECT 'primary_device', primary_device, COUNT(*) FROM u GROUP BY 2
UNION ALL
SELECT 'is_test_account_flag', CAST(is_test_account_flag AS STRING), COUNT(*) FROM u GROUP BY 2
ORDER BY column_name, users DESC;

column_name,value,users
acquisition_channel,email_campaign,201
acquisition_channel,organic_search,189
acquisition_channel,paid_social,188
acquisition_channel,referral,162
acquisition_channel,NULL,20
acquisition_channel,influencer_TEST_CAMPAIGN,20
acquisition_channel,carrier_pigeon,20
age_band,18-24,121
age_band,25-34,117
age_band,13-17,111


In [0]:
WITH u AS (
  SELECT DISTINCT * FROM workspace.scribd_exercise.exercise_users
)
SELECT CASE WHEN acquisition_channel = 'influencer_TEST_CAMPAIGN' THEN 'yes' ELSE 'no' END AS test_campaign,
       CASE WHEN acquisition_channel = 'carrier_pigeon' THEN 'yes' ELSE 'no' END AS carrier_pigeon,
       CASE WHEN primary_device = 'smart_toaster' THEN 'yes' ELSE 'no' END AS smart_toaster,
       CASE WHEN primary_device = 'unknown_device' THEN 'yes' ELSE 'no' END AS unknown_device,
       CASE WHEN age_band = '999' THEN 'yes' ELSE 'no' END AS age_999,
       is_test_account_flag,
       COUNT(*) AS users
FROM u
WHERE acquisition_channel IN ('influencer_TEST_CAMPAIGN', 'carrier_pigeon')
   OR primary_device IN ('smart_toaster', 'unknown_device')
   OR age_band = '999'
GROUP BY 1, 2, 3, 4, 5, 6
ORDER BY users DESC;

test_campaign,carrier_pigeon,smart_toaster,unknown_device,age_999,is_test_account_flag,users
yes,no,no,no,no,false,20
no,yes,no,no,no,false,19
no,no,no,yes,no,false,18
no,no,yes,no,no,false,16
no,no,no,no,yes,false,6
no,no,no,yes,no,true,2
no,no,yes,no,no,true,2
no,no,no,no,yes,true,1
no,yes,yes,no,no,false,1
no,no,yes,no,yes,false,1


**Findings:**
- influencer_TEST_CAMPAIGN (20 users): all regular accounts with no other anomalies, leaning towards a real marketing test, not test data. Kept as its own channel.
- Invalid values: age_band '999' (8), carrier_pigeon (20), unknown_device (20), smart_toaster (20). Spread across different users (only 2 have two), and 61 of 66 are regular accounts: independent field problems.
- 13–17 age band: 111 users (13.9%). Not an error, but a caveat for subscriptions and engagement.

**Treatment:** map invalid values to 'Unknown' and flag invalid_segment_value.

## Test accounts - How many test accounts are there? How much activity do they have? 

In [0]:
-- Who are the test accounts? 
WITH u AS (
  SELECT user_id,
         MAX(CAST(is_test_account_flag AS INT)) AS is_test,
         MIN(signup_date) AS signup_date
  FROM workspace.scribd_exercise.exercise_users
  GROUP BY user_id
)
SELECT CASE WHEN is_test = 1 THEN 'test account' ELSE 'regular account' END AS account_type,
       COUNT(*) AS users,
       MIN(user_id) AS lowest_id,
       MAX(user_id) AS highest_id,
       MIN(signup_date) AS first_signup,
       MAX(signup_date) AS last_signup
FROM u
GROUP BY 1;

account_type,users,lowest_id,highest_id,first_signup,last_signup
regular account,760,USR_00001,USR_00800,2021-01-03,2026-09-16
test account,40,USR_00040,USR_00743,2021-05-14,2026-07-31


In [0]:
-- How much engagement comes from them? 
WITH u AS (
  SELECT user_id, MAX(CAST(is_test_account_flag AS INT)) AS is_test
  FROM workspace.scribd_exercise.exercise_users
  GROUP BY user_id
)
SELECT CASE WHEN u.user_id IS NULL THEN 'user not in users table'
            WHEN u.is_test = 1 THEN 'test account'
            ELSE 'regular account' END AS account_type,
       COUNT(*) AS event_rows,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct_of_events,
       COUNT(DISTINCT e.user_id) AS users
FROM workspace.scribd_exercise.exercise_engagement_events e
LEFT JOIN u ON e.user_id = u.user_id
GROUP BY 1
ORDER BY event_rows DESC;

account_type,event_rows,pct_of_events,users
regular account,5692,94.87,759
test account,308,5.13,40


In [0]:
-- Engagement from test accounts by business
WITH u AS (
  SELECT user_id, MAX(CAST(is_test_account_flag AS INT)) AS is_test
  FROM workspace.scribd_exercise.exercise_users
  GROUP BY user_id
)
SELECT COALESCE(ci.business_id, 'no content') AS business_id,
       COUNT(*) AS event_rows,
       SUM(CASE WHEN u.is_test = 1 THEN 1 ELSE 0 END) AS test_account_events,
       ROUND(100.0 * SUM(CASE WHEN u.is_test = 1 THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_test
FROM workspace.scribd_exercise.exercise_engagement_events e
LEFT JOIN u ON e.user_id = u.user_id
LEFT JOIN workspace.scribd_exercise.exercise_content_items ci ON e.content_id = ci.content_id
GROUP BY 1
ORDER BY 1;

business_id,event_rows,test_account_events,pct_test
BIZ_1,2078,108,5.20
BIZ_2,1627,72,4.43
BIZ_3,1972,109,5.53
no content,323,19,5.88


In [0]:
-- Subscriptions from test accounts
WITH u AS (
  SELECT user_id, MAX(CAST(is_test_account_flag AS INT)) AS is_test
  FROM workspace.scribd_exercise.exercise_users
  GROUP BY user_id
)
SELECT CASE WHEN u.user_id IS NULL THEN 'user not in users table'
            WHEN u.is_test = 1 THEN 'test account'
            ELSE 'regular account' END AS account_type,
       COUNT(*) AS subscriptions,
       COUNT(DISTINCT s.user_id) AS users,
       SUM(CASE WHEN s.status = 'TEST_STATUS_DO_NOT_USE' THEN 1 ELSE 0 END) AS test_status,
       ROUND(SUM(s.monthly_price_usd), 2) AS total_monthly_price
FROM workspace.scribd_exercise.exercise_subscriptions s
LEFT JOIN u ON s.user_id = u.user_id
GROUP BY 1
ORDER BY subscriptions DESC;

account_type,subscriptions,users,test_status,total_monthly_price
regular account,804,639,19,15942.18
test account,39,33,1,461.7


**Findings (4a–4d):** 
- 40 test accounts (5.0% of users), spread across IDs and signup dates from 2021 to 2026, so created continuously, not in one batch. 
- All are active: 308 events (5.1% of all events; 4.4%–5.9% in every business) and 39 subscriptions from 33 users (4.6%). Every event and subscription links to an existing user. 
- TEST_STATUS_DO_NOT_USE is mostly on regular accounts (19 of 20), a separate problem for the subscriptions audit.

**Treatment:** quarantine the 40 test users and all their events and subscriptions (test_account)

**Impact:** users 760 (not 800), events 5,692 (not 6,000), subscriptions 804 (not 843)

### Date ranges

In [0]:
-- Signup date range 
SELECT MIN(signup_date) AS earliest_signup,
       MAX(signup_date) AS latest_signup,
       SUM(CASE WHEN signup_date > CURRENT_DATE() THEN 1 ELSE 0 END) AS future_dates,
       SUM(CASE WHEN signup_date < DATE'2021-01-01' THEN 1 ELSE 0 END) AS before_first_business_launch
FROM (
  SELECT user_id, MIN(signup_date) AS signup_date
  FROM workspace.scribd_exercise.exercise_users
  GROUP BY user_id
);

earliest_signup,latest_signup,future_dates,before_first_business_launch
2021-01-03,2026-09-16,0,0


In [0]:
-- Activity before signup 
WITH u AS (
  SELECT user_id,
         MIN(signup_date) AS signup_date,
         MAX(CAST(is_test_account_flag AS INT)) AS is_test
  FROM workspace.scribd_exercise.exercise_users
  GROUP BY user_id
)
SELECT 'events' AS activity,
       COUNT(*) AS rows_before_signup,
       COUNT(DISTINCT e.user_id) AS users,
       SUM(CASE WHEN u.is_test = 1 THEN 1 ELSE 0 END) AS from_test_accounts,
       MIN(DATEDIFF(u.signup_date, CAST(e.event_timestamp AS DATE))) AS min_days_early,
       MAX(DATEDIFF(u.signup_date, CAST(e.event_timestamp AS DATE))) AS max_days_early
FROM workspace.scribd_exercise.exercise_engagement_events e
JOIN u ON e.user_id = u.user_id
WHERE CAST(e.event_timestamp AS DATE) < u.signup_date
UNION ALL
SELECT 'subscriptions',
       COUNT(*),
       COUNT(DISTINCT s.user_id),
       SUM(CASE WHEN u.is_test = 1 THEN 1 ELSE 0 END),
       MIN(DATEDIFF(u.signup_date, s.start_date)),
       MAX(DATEDIFF(u.signup_date, s.start_date))
FROM workspace.scribd_exercise.exercise_subscriptions s
JOIN u ON s.user_id = u.user_id
WHERE s.start_date < u.signup_date;

activity,rows_before_signup,users,from_test_accounts,min_days_early,max_days_early
events,30,28,0,2,31
subscriptions,15,15,0,3,59


**Findings:** 
- 30 events from 28 users (2–31 days early) 
- 15 subscriptions from 15 users (3–59 days early) are dated before signup. 

**Treatment:** events up to 30 days early keep + flag activity_before_signup; more than 30 days early quarantine; subscriptions keep + flag started_before_signup.

# Subscriptions table

### Uniqueness - is each subscription listed once? 

In [0]:
SELECT COUNT(*) AS total_rows,
       COUNT(DISTINCT subscription_id) AS distinct_ids,
       COUNT(*) - COUNT(DISTINCT subscription_id) AS extra_rows
FROM workspace.scribd_exercise.exercise_subscriptions;

total_rows,distinct_ids,extra_rows
843,843,0


### Completeness - Which columns have missing values? 

In [0]:
SELECT 'subscription_id' AS column_name,
       SUM(CASE WHEN subscription_id IS NULL OR TRIM(subscription_id) = '' THEN 1 ELSE 0 END) AS missing,
       SUM(CASE WHEN LOWER(TRIM(subscription_id)) IN ('n/a','na','null','none','unknown','-') THEN 1 ELSE 0 END) AS placeholders
FROM workspace.scribd_exercise.exercise_subscriptions
UNION ALL
SELECT 'user_id',
       SUM(CASE WHEN user_id IS NULL OR TRIM(user_id) = '' THEN 1 ELSE 0 END),
       SUM(CASE WHEN LOWER(TRIM(user_id)) IN ('n/a','na','null','none','unknown','-') THEN 1 ELSE 0 END)
FROM workspace.scribd_exercise.exercise_subscriptions
UNION ALL
SELECT 'business_id',
       SUM(CASE WHEN business_id IS NULL OR TRIM(business_id) = '' THEN 1 ELSE 0 END),
       SUM(CASE WHEN LOWER(TRIM(business_id)) IN ('n/a','na','null','none','unknown','-') THEN 1 ELSE 0 END)
FROM workspace.scribd_exercise.exercise_subscriptions
UNION ALL
SELECT 'plan_tier',
       SUM(CASE WHEN plan_tier IS NULL OR TRIM(plan_tier) = '' THEN 1 ELSE 0 END),
       SUM(CASE WHEN LOWER(TRIM(plan_tier)) IN ('n/a','na','null','none','unknown','-') THEN 1 ELSE 0 END)
FROM workspace.scribd_exercise.exercise_subscriptions
UNION ALL
SELECT 'billing_cycle',
       SUM(CASE WHEN billing_cycle IS NULL OR TRIM(billing_cycle) = '' THEN 1 ELSE 0 END),
       SUM(CASE WHEN LOWER(TRIM(billing_cycle)) IN ('n/a','na','null','none','unknown','-') THEN 1 ELSE 0 END)
FROM workspace.scribd_exercise.exercise_subscriptions
UNION ALL
SELECT 'status',
       SUM(CASE WHEN status IS NULL OR TRIM(status) = '' THEN 1 ELSE 0 END),
       SUM(CASE WHEN LOWER(TRIM(status)) IN ('n/a','na','null','none','unknown','-') THEN 1 ELSE 0 END)
FROM workspace.scribd_exercise.exercise_subscriptions
UNION ALL
SELECT 'monthly_price_usd', SUM(CASE WHEN monthly_price_usd IS NULL THEN 1 ELSE 0 END), 0
FROM workspace.scribd_exercise.exercise_subscriptions
UNION ALL
SELECT 'start_date', SUM(CASE WHEN start_date IS NULL THEN 1 ELSE 0 END), 0
FROM workspace.scribd_exercise.exercise_subscriptions
UNION ALL
SELECT 'end_date', SUM(CASE WHEN end_date IS NULL THEN 1 ELSE 0 END), 0
FROM workspace.scribd_exercise.exercise_subscriptions;

column_name,missing,placeholders
subscription_id,0,0
user_id,0,0
business_id,0,0
plan_tier,0,0
billing_cycle,215,0
status,0,0
monthly_price_usd,0,0
start_date,0,0
end_date,619,0


In [0]:
-- Are missing end_date consistent with the status?
SELECT status,
       COUNT(*) AS subscriptions,
       SUM(CASE WHEN end_date IS NULL THEN 1 ELSE 0 END) AS no_end_date,
       SUM(CASE WHEN end_date IS NOT NULL THEN 1 ELSE 0 END) AS has_end_date
FROM workspace.scribd_exercise.exercise_subscriptions
GROUP BY status
ORDER BY subscriptions DESC;

status,subscriptions,no_end_date,has_end_date
active,465,465,0
canceled,224,0,224
paused,96,96,0
trialing,38,38,0
TEST_STATUS_DO_NOT_USE,20,20,0


In [0]:
-- where is the billing cycle missing? 
SELECT business_id,
       plan_tier,
       COUNT(*) AS subscriptions,
       SUM(CASE WHEN billing_cycle IS NULL OR TRIM(billing_cycle) = '' THEN 1 ELSE 0 END) AS missing_billing_cycle
FROM workspace.scribd_exercise.exercise_subscriptions
GROUP BY business_id, plan_tier
ORDER BY business_id, plan_tier;

business_id,plan_tier,subscriptions,missing_billing_cycle
BIZ_1,Basic,69,0
BIZ_1,Free,71,71
BIZ_1,Premium,60,0
BIZ_1,Premium_Plus_Deluxe_XL,72,0
BIZ_2,Basic,69,0
BIZ_2,Free,70,70
BIZ_2,Premium,68,0
BIZ_2,Premium_Plus_Deluxe_XL,90,0
BIZ_3,Basic,77,0
BIZ_3,Free,74,74


**Findings:** 
- Only billing_cycle (215) and end_date (619) have missing values, and both are explained:
- end_date matches status: all 224 canceled subscriptions have one; no active, paused, trialing or test-status subscription doesn't.
- billing_cycle is missing exactly for the 215 Free-plan subscriptions and present for all paid plans. 


### Valid categories

In [0]:
SELECT 'status' AS column_name, status AS value, COUNT(*) AS subscriptions
FROM workspace.scribd_exercise.exercise_subscriptions GROUP BY 2
UNION ALL
SELECT 'plan_tier', plan_tier, COUNT(*)
FROM workspace.scribd_exercise.exercise_subscriptions GROUP BY 2
UNION ALL
SELECT 'billing_cycle', COALESCE(billing_cycle, 'NULL'), COUNT(*)
FROM workspace.scribd_exercise.exercise_subscriptions GROUP BY 2
UNION ALL
SELECT 'monthly_price_usd', CAST(monthly_price_usd AS STRING), COUNT(*)
FROM workspace.scribd_exercise.exercise_subscriptions GROUP BY 2
ORDER BY column_name, subscriptions DESC;

column_name,value,subscriptions
billing_cycle,annual,259
billing_cycle,monthly,221
billing_cycle,NULL,215
billing_cycle,lifetime_deal_fake,148
monthly_price_usd,0.0,215
monthly_price_usd,14.99,124
monthly_price_usd,9.99,111
monthly_price_usd,7.99,102
monthly_price_usd,24.99,84
monthly_price_usd,11.99,77


In [0]:
-- what does the test-status subscriptions look like? 
SELECT business_id,
       plan_tier,
       COALESCE(billing_cycle, 'NULL') AS billing_cycle,
       monthly_price_usd,
       COUNT(*) AS subscriptions,
       MIN(start_date) AS first_start,
       MAX(start_date) AS last_start
FROM workspace.scribd_exercise.exercise_subscriptions
WHERE status = 'TEST_STATUS_DO_NOT_USE'
GROUP BY business_id, plan_tier, COALESCE(billing_cycle, 'NULL'), monthly_price_usd
ORDER BY business_id, plan_tier;

business_id,plan_tier,billing_cycle,monthly_price_usd,subscriptions,first_start,last_start
BIZ_1,Basic,annual,7.99,1,2026-04-04,2026-04-04
BIZ_1,Free,NULL,0.0,2,2023-12-06,2026-05-25
BIZ_1,Premium,lifetime_deal_fake,9.99,2,2026-02-08,2026-06-22
BIZ_2,Basic,lifetime_deal_fake,4.99,1,2026-05-21,2026-05-21
BIZ_2,Basic,monthly,9.99,1,2023-12-06,2023-12-06
BIZ_2,Free,NULL,0.0,2,2025-10-24,2026-04-16
BIZ_2,Premium,monthly,14.99,1,2026-07-08,2026-07-08
BIZ_2,Premium,annual,11.99,1,2025-08-19,2025-08-19
BIZ_2,Premium_Plus_Deluxe_XL,monthly,24.99,1,2023-10-17,2023-10-17
BIZ_3,Basic,annual,7.99,2,2026-08-08,2026-09-02


In [0]:
-- How does activity of test status looks like?
SELECT s.subscription_id,
       s.business_id,
       s.start_date,
       COUNT(e.event_id) AS events_after_start,
       MAX(CAST(e.event_timestamp AS DATE)) AS last_event
FROM workspace.scribd_exercise.exercise_subscriptions s
LEFT JOIN workspace.scribd_exercise.exercise_engagement_events e`   `
  ON e.user_id = s.user_id
 AND CAST(e.event_timestamp AS DATE) >= s.start_date
WHERE s.status = 'TEST_STATUS_DO_NOT_USE'
GROUP BY s.subscription_id, s.business_id, s.start_date
ORDER BY events_after_start;

subscription_id,business_id,start_date,events_after_start,last_event
SUB_000064,BIZ_2,2026-07-08,1,2026-08-10
SUB_000790,BIZ_3,2026-08-08,1,2026-08-13
SUB_000560,BIZ_3,2026-09-02,1,2026-09-14
SUB_000307,BIZ_1,2026-04-04,2,2026-08-28
SUB_000120,BIZ_3,2026-02-03,2,2026-08-04
SUB_000156,BIZ_3,2026-02-10,3,2026-08-21
SUB_000772,BIZ_1,2026-06-22,3,2026-08-22
SUB_000302,BIZ_2,2025-08-19,3,2026-09-02
SUB_000650,BIZ_2,2026-05-21,3,2026-07-21
SUB_000358,BIZ_1,2026-02-08,3,2026-09-16


In [0]:
-- How does activity of test status looks: own business, other business, no content?
SELECT s.subscription_id,
       s.business_id,
       s.plan_tier,
       s.start_date,
       COUNT(CASE WHEN ci.business_id = s.business_id THEN e.event_id END) AS events_same_business,
       MAX(CASE WHEN ci.business_id = s.business_id THEN CAST(e.event_timestamp AS DATE) END) AS last_event_same_business,
       COUNT(CASE WHEN ci.business_id <> s.business_id THEN e.event_id END) AS events_other_business,
       COUNT(CASE WHEN e.event_id IS NOT NULL AND e.content_id IS NULL THEN e.event_id END) AS events_no_content
FROM workspace.scribd_exercise.exercise_subscriptions s
LEFT JOIN workspace.scribd_exercise.exercise_engagement_events e
  ON e.user_id = s.user_id
 AND CAST(e.event_timestamp AS DATE) >= s.start_date
LEFT JOIN workspace.scribd_exercise.exercise_content_items ci
  ON e.content_id = ci.content_id
WHERE s.status = 'TEST_STATUS_DO_NOT_USE'
GROUP BY s.subscription_id, s.business_id, s.plan_tier, s.start_date
ORDER BY events_same_business;

subscription_id,business_id,plan_tier,start_date,events_same_business,last_event_same_business,events_other_business,events_no_content
SUB_000120,BIZ_3,Basic,2026-02-03,0,null,2,0
SUB_000560,BIZ_3,Basic,2026-09-02,0,null,0,1
SUB_000302,BIZ_2,Premium,2025-08-19,0,null,3,0
SUB_000772,BIZ_1,Premium,2026-06-22,0,null,3,0
SUB_000064,BIZ_2,Premium,2026-07-08,0,null,1,0
SUB_000650,BIZ_2,Basic,2026-05-21,1,2026-06-30,2,0
SUB_000790,BIZ_3,Basic,2026-08-08,1,2026-08-13,0,0
SUB_000540,BIZ_3,Premium_Plus_Deluxe_XL,2024-11-27,1,2026-09-10,2,1
SUB_000156,BIZ_3,Premium,2026-02-10,1,2026-08-21,2,0
SUB_000362,BIZ_1,Free,2023-12-06,2,2026-08-29,6,1


In [0]:
-- Are accounts with test status paying?
SELECT s.subscription_id,
       s.business_id,
       s.plan_tier,
       COALESCE(s.billing_cycle, 'NULL') AS billing_cycle,
       s.monthly_price_usd AS listed_monthly_price,
       COLLECT_SET(e.event_type) AS event_types_after_start,
       SUM(CASE WHEN e.event_type LIKE 'subscription%' THEN 1 ELSE 0 END) AS subscription_events
FROM workspace.scribd_exercise.exercise_subscriptions s
LEFT JOIN workspace.scribd_exercise.exercise_engagement_events e
  ON e.user_id = s.user_id
 AND CAST(e.event_timestamp AS DATE) >= s.start_date
WHERE s.status = 'TEST_STATUS_DO_NOT_USE'
GROUP BY s.subscription_id, s.business_id, s.plan_tier, COALESCE(s.billing_cycle, 'NULL'), s.monthly_price_usd
ORDER BY listed_monthly_price DESC;

subscription_id,business_id,plan_tier,billing_cycle,listed_monthly_price,event_types_after_start,subscription_events
SUB_000433,BIZ_2,Premium_Plus_Deluxe_XL,monthly,24.99,"List(download, content_complete, content_open, like, search_query, subscription_start)",1
SUB_000064,BIZ_2,Premium,monthly,14.99,List(share),0
SUB_000540,BIZ_3,Premium_Plus_Deluxe_XL,lifetime_deal_fake,14.99,"List(content_complete, content_open, page_view)",0
SUB_000831,BIZ_3,Premium_Plus_Deluxe_XL,lifetime_deal_fake,14.99,"List(content_complete, content_open, download, comment_post)",0
SUB_000302,BIZ_2,Premium,annual,11.99,"List(download, share, content_complete)",0
SUB_000120,BIZ_3,Basic,monthly,9.99,List(content_complete),0
SUB_000156,BIZ_3,Premium,lifetime_deal_fake,9.99,"List(download, content_complete)",0
SUB_000299,BIZ_2,Basic,monthly,9.99,"List(share, page_view, comment_post, content_complete, content_open)",0
SUB_000358,BIZ_1,Premium,lifetime_deal_fake,9.99,"List(comment_post, download)",0
SUB_000772,BIZ_1,Premium,lifetime_deal_fake,9.99,"List(download, content_complete)",0


In [0]:
-- Are the 5 paid test-status customers with no activity on their own business unusual?
SELECT CASE WHEN s.status = 'TEST_STATUS_DO_NOT_USE' THEN 'test status' ELSE s.status END AS status_group,
       COUNT(*) AS subscriptions,
       SUM(CASE WHEN own_events = 0 THEN 1 ELSE 0 END) AS no_own_business_events,
       ROUND(100.0 * SUM(CASE WHEN own_events = 0 THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_no_own_events
FROM (
  SELECT s.subscription_id, s.status,
         COUNT(CASE WHEN ci.business_id = s.business_id THEN e.event_id END) AS own_events
  FROM workspace.scribd_exercise.exercise_subscriptions s
  LEFT JOIN workspace.scribd_exercise.exercise_engagement_events e
    ON e.user_id = s.user_id
   AND CAST(e.event_timestamp AS DATE) >= s.start_date
  LEFT JOIN workspace.scribd_exercise.exercise_content_items ci
    ON e.content_id = ci.content_id
  WHERE s.plan_tier <> 'Free'
  GROUP BY s.subscription_id, s.status
) s
GROUP BY 1
ORDER BY 1;

status_group,subscriptions,no_own_business_events,pct_no_own_events
active,324,99,30.6
canceled,189,48,25.4
paused,70,23,32.9
test status,15,5,33.3
trialing,30,8,26.7


**Findings:**
- **20 subscriptions (2.4% of 843)** have status TEST_STATUS_DO_NOT_USE.
- **They look like real subscriptions with a bad status, not test data:**
  - 19 of 20 belong to regular accounts (1 to a test account).
  - Spread across all 3 businesses, all 4 plans and all billing cycles, with normal prices and start dates from 2023 to 2026.
  - None has an end date, while every canceled subscription does, so they are very likely not canceled (active, paused or trialing, but I can't tell which).
- **The customers are still using the product:** all 20 had activity after their start date, with last activity between June and September 2026. 15 of 20 used the business they subscribed to.
- **No sign of lost access:** 5 of 15 paid ones (33%) had no activity on their own business, the same as normal active paid subscribers (31%).
- **Payment can't be verified:** the data has no payment, renewal or invoice events. 15 of the 20 are paid plans listing **$165.85/month**; 5 are Free.

**Treatment**
- **Keep + flag** in the clean table: status set to 'unknown', dq_flag = invalid_status. The 1 on a test account is quarantined with the test accounts (issue #14).
- **Excluded** from active subscribers, MRR and churn.
- **Reported as a range:** active subscribers up to +19 and MRR up to about +$166/month, since these customers are likely still active.

**Recommendation**
- Escalate to engineering: verify billing and access for these customers, and stop test values from being written to production records.
- Analytics severity: Low (small and contained). Operational risk: worth investigating, alongside the test content shown to real customers.

In [0]:
-- what do lifetime deals look like? 
SELECT business_id,
       plan_tier,
       status,
       COUNT(*) AS subscriptions,
       MIN(monthly_price_usd) AS min_price,
       MAX(monthly_price_usd) AS max_price,
       MIN(start_date) AS first_start,
       MAX(start_date) AS last_start
FROM workspace.scribd_exercise.exercise_subscriptions
WHERE billing_cycle = 'lifetime_deal_fake'
GROUP BY business_id, plan_tier, status
ORDER BY business_id, plan_tier, status;

business_id,plan_tier,status,subscriptions,min_price,max_price,first_start,last_start
BIZ_1,Basic,active,10,-9.99,999.99,2023-05-16,2026-08-29
BIZ_1,Basic,canceled,2,4.99,4.99,2024-03-03,2026-03-26
BIZ_1,Basic,trialing,1,4.99,4.99,2026-06-02,2026-06-02
BIZ_1,Premium,TEST_STATUS_DO_NOT_USE,2,9.99,9.99,2026-02-08,2026-06-22
BIZ_1,Premium,active,10,-9.99,9.99,2023-12-17,2026-09-12
BIZ_1,Premium,canceled,4,9.99,9.99,2023-10-20,2026-06-25
BIZ_1,Premium_Plus_Deluxe_XL,active,15,14.99,14.99,2022-03-01,2026-09-19
BIZ_1,Premium_Plus_Deluxe_XL,canceled,1,14.99,14.99,2022-06-11,2022-06-11
BIZ_1,Premium_Plus_Deluxe_XL,paused,3,14.99,14.99,2025-05-05,2026-04-23
BIZ_1,Premium_Plus_Deluxe_XL,trialing,1,14.99,14.99,2026-08-11,2026-08-11


In [0]:
-- How do lifetime value deals' prices compare to other plans
SELECT plan_tier,
       MAX(CASE WHEN billing_cycle = 'monthly' THEN typical_price END) AS monthly_price,
       MAX(CASE WHEN billing_cycle = 'annual' THEN typical_price END) AS annual_price,
       MAX(CASE WHEN billing_cycle = 'lifetime_deal_fake' THEN typical_price END) AS lifetime_price,
       SUM(CASE WHEN billing_cycle = 'monthly' THEN subscriptions END) AS monthly_subs,
       SUM(CASE WHEN billing_cycle = 'annual' THEN subscriptions END) AS annual_subs,
       SUM(CASE WHEN billing_cycle = 'lifetime_deal_fake' THEN subscriptions END) AS lifetime_subs,
       SUM(placeholder_prices) AS placeholder_prices
FROM (
  SELECT plan_tier,
         billing_cycle,
         COUNT(*) AS subscriptions,
         MODE(CASE WHEN monthly_price_usd NOT IN (-9.99, 999.99) THEN monthly_price_usd END) AS typical_price,
         SUM(CASE WHEN monthly_price_usd IN (-9.99, 999.99) THEN 1 ELSE 0 END) AS placeholder_prices
  FROM workspace.scribd_exercise.exercise_subscriptions
  WHERE plan_tier <> 'Free'
  GROUP BY plan_tier, billing_cycle
)
GROUP BY plan_tier
ORDER BY monthly_price;

plan_tier,monthly_price,annual_price,lifetime_price,monthly_subs,annual_subs,lifetime_subs,placeholder_prices
Basic,9.99,7.99,4.99,69,104,42,5
Premium,14.99,11.99,9.99,65,78,45,4
Premium_Plus_Deluxe_XL,24.99,19.99,14.99,87,77,61,7


In [0]:
SELECT COALESCE(billing_cycle, 'NULL (Free)') AS billing_cycle,
       COUNT(*) AS subscriptions,
       ROUND(100.0 * SUM(CASE WHEN status = 'canceled' THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_canceled,
       ROUND(100.0 * SUM(CASE WHEN status = 'paused' THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_paused,
       ROUND(100.0 * SUM(CASE WHEN status = 'trialing' THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_trialing,
       ROUND(AVG(DATEDIFF(COALESCE(end_date, DATE'2026-09-19'), start_date)), 0) AS avg_days_subscribed,
       MIN(start_date) AS first_start,
       MAX(start_date) AS last_start,
       SUM(CASE WHEN monthly_price_usd IN (-9.99, 999.99) THEN 1 ELSE 0 END) AS placeholder_prices
FROM workspace.scribd_exercise.exercise_subscriptions
GROUP BY 1
ORDER BY subscriptions DESC;

billing_cycle,subscriptions,pct_canceled,pct_paused,pct_trialing,avg_days_subscribed,first_start,last_start,placeholder_prices
annual,259,22.8,16.2,5.8,475.0,2021-07-24,2026-09-19,6
monthly,221,50.2,7.2,4.5,396.0,2021-12-08,2026-09-19,5
NULL (Free),215,16.3,12.1,3.7,425.0,2021-12-08,2026-09-19,0
lifetime_deal_fake,148,12.8,8.1,3.4,450.0,2021-12-01,2026-09-19,5


**Findings:** 
- Each paid plan has exactly one price per billing cycle, always ordered monthly > annual > lifetime (Basic 9.99 / 7.99 / 4.99; Premium 14.99 / 11.99 / 9.99; Premium_Plus_Deluxe_XL 24.99 / 19.99 / 14.99). 
- Lifetime is the deepest discount (33–50% off monthly), consistent with a price locked for life. All prices are monthly amounts, including annual, so MRR can sum monthly_price_usd directly. 
- 148 of 628 paid subscriptions (24%) are lifetime deals.

**Treatment (proposed):** include lifetime deals in MRR at their listed monthly price. Assumption: lifetime_deal_fake is a locked-in monthly price, not a one-time payment (confirm with the data owner). 

### Prices - which ssuscriptions have impossible prices? What should the price be?

In [0]:
-- Placeholder value in detail 
SELECT subscription_id, user_id, business_id, plan_tier, billing_cycle, status,
       monthly_price_usd, start_date, end_date
FROM workspace.scribd_exercise.exercise_subscriptions
WHERE monthly_price_usd IN (-9.99, 999.99)
ORDER BY monthly_price_usd, business_id, plan_tier;

subscription_id,user_id,business_id,plan_tier,billing_cycle,status,monthly_price_usd,start_date,end_date
SUB_000132,USR_00130,BIZ_1,Basic,lifetime_deal_fake,active,-9.99,2025-06-13,null
SUB_000463,USR_00443,BIZ_1,Premium,lifetime_deal_fake,active,-9.99,2026-07-30,null
SUB_000205,USR_00187,BIZ_1,Premium_Plus_Deluxe_XL,monthly,active,-9.99,2025-11-09,null
SUB_000071,USR_00069,BIZ_2,Premium,monthly,active,-9.99,2026-03-15,null
SUB_000455,USR_00434,BIZ_2,Premium_Plus_Deluxe_XL,annual,trialing,-9.99,2024-04-16,null
SUB_000797,USR_00760,BIZ_3,Basic,annual,active,-9.99,2025-06-29,null
SUB_000780,USR_00746,BIZ_3,Premium_Plus_Deluxe_XL,monthly,active,-9.99,2026-02-04,null
SUB_000308,USR_00290,BIZ_3,Premium_Plus_Deluxe_XL,annual,active,-9.99,2024-04-27,null
SUB_000227,USR_00203,BIZ_1,Basic,monthly,paused,999.99,2023-11-23,null
SUB_000491,USR_00467,BIZ_1,Basic,annual,active,999.99,2023-01-17,null


In [0]:
-- Does every other subscription match the price grid?
WITH grid AS (
  SELECT plan_tier, billing_cycle,
         MODE(monthly_price_usd) AS expected_price
  FROM workspace.scribd_exercise.exercise_subscriptions
  WHERE monthly_price_usd NOT IN (-9.99, 999.99)
  GROUP BY plan_tier, billing_cycle
)
SELECT CASE WHEN s.monthly_price_usd IN (-9.99, 999.99) THEN 'placeholder'
            WHEN s.monthly_price_usd = g.expected_price THEN 'matches grid'
            ELSE 'other mismatch' END AS price_check,
       COUNT(*) AS subscriptions
FROM workspace.scribd_exercise.exercise_subscriptions s
JOIN grid g
  ON s.plan_tier = g.plan_tier
 AND COALESCE(s.billing_cycle, 'NULL') = COALESCE(g.billing_cycle, 'NULL')
GROUP BY 1;

price_check,subscriptions
placeholder,16
matches grid,827


**FindingS:** 
- 16 subscriptions (1.9%) have placeholder prices (8 at 999.99, 8 at -9.99), spread across businesses, plans, billing cycles and 2022–2026; 13 are active. 

**Impact:** for the 13 active ones, raw prices add $5,930.01/month to MRR vs. $177.87 at grid prices, an overstatement of about $5,752/month.

**Treatment:** impute the grid price for each subscription's plan and billing cycle; flag price_imputed_from_grid.

### Dates consistency - are subscription dates in a logical order and range? Fo long-running statuses (like trials) make sense? 

In [0]:
-- Date logic
SELECT SUM(CASE WHEN end_date < start_date THEN 1 ELSE 0 END) AS ends_before_start,
       SUM(CASE WHEN end_date = start_date THEN 1 ELSE 0 END) AS ends_same_day,
       SUM(CASE WHEN start_date > CURRENT_DATE() THEN 1 ELSE 0 END) AS future_start,
       SUM(CASE WHEN end_date > CURRENT_DATE() THEN 1 ELSE 0 END) AS future_end,
       SUM(CASE WHEN end_date > DATE'2026-09-19' THEN 1 ELSE 0 END) AS end_after_data_end,
       MIN(start_date) AS first_start,
       MAX(start_date) AS last_start,
       MIN(end_date) AS first_end,
       MAX(end_date) AS last_end
FROM workspace.scribd_exercise.exercise_subscriptions;

ends_before_start,ends_same_day,future_start,future_end,end_after_data_end,first_start,last_start,first_end,last_end
0,0,0,0,0,2021-07-24,2026-09-19,2022-07-31,2026-09-19


In [0]:
-- How long have trials and pauses lasted? 
SELECT status,
       COUNT(*) AS subscriptions,
       MIN(DATEDIFF(DATE'2026-09-19', start_date)) AS min_days_since_start,
       PERCENTILE(DATEDIFF(DATE'2026-09-19', start_date), 0.5) AS median_days_since_start,
       MAX(DATEDIFF(DATE'2026-09-19', start_date)) AS max_days_since_start,
       SUM(CASE WHEN DATEDIFF(DATE'2026-09-19', start_date) > 30 THEN 1 ELSE 0 END) AS over_30_days
FROM workspace.scribd_exercise.exercise_subscriptions
WHERE status IN ('trialing', 'paused', 'active')
GROUP BY status
ORDER BY status;

status,subscriptions,min_days_since_start,median_days_since_start,max_days_since_start,over_30_days
active,465,0,378.0,1753,423
paused,96,0,390.0,1735,85
trialing,38,1,267.0,1473,33


In [0]:
SELECT business_id,
       plan_tier,
       COALESCE(billing_cycle, 'NULL') AS billing_cycle,
       monthly_price_usd,
       COUNT(*) AS trialing,
       SUM(CASE WHEN DATEDIFF(DATE'2026-09-19', start_date) > 30 THEN 1 ELSE 0 END) AS stale_over_30_days
FROM workspace.scribd_exercise.exercise_subscriptions
WHERE status = 'trialing'
GROUP BY business_id, plan_tier, COALESCE(billing_cycle, 'NULL'), monthly_price_usd
ORDER BY business_id, plan_tier, billing_cycle;

business_id,plan_tier,billing_cycle,monthly_price_usd,trialing,stale_over_30_days
BIZ_1,Basic,annual,7.99,1,1
BIZ_1,Basic,lifetime_deal_fake,4.99,1,1
BIZ_1,Free,NULL,0.0,2,2
BIZ_1,Premium,annual,11.99,3,2
BIZ_1,Premium,monthly,14.99,1,1
BIZ_1,Premium_Plus_Deluxe_XL,annual,19.99,1,1
BIZ_1,Premium_Plus_Deluxe_XL,lifetime_deal_fake,14.99,1,1
BIZ_1,Premium_Plus_Deluxe_XL,monthly,24.99,2,1
BIZ_2,Basic,monthly,9.99,2,2
BIZ_2,Free,NULL,0.0,4,3


In [0]:
SELECT CASE WHEN status = 'trialing' AND DATEDIFF(DATE'2026-09-19', start_date) > 30 THEN 'stale trial'
            WHEN status = 'trialing' THEN 'recent trial'
            ELSE status END AS status_group,
       COUNT(*) AS subscriptions,
       SUM(CASE WHEN recent_events > 0 THEN 1 ELSE 0 END) AS active_last_90_days,
       ROUND(100.0 * SUM(CASE WHEN recent_events > 0 THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_active_last_90_days,
       ROUND(SUM(monthly_price_usd), 2) AS listed_monthly_price
FROM (
  SELECT s.subscription_id, s.status, s.start_date, s.monthly_price_usd,
         COUNT(e.event_id) AS recent_events
  FROM workspace.scribd_exercise.exercise_subscriptions s
  LEFT JOIN workspace.scribd_exercise.exercise_engagement_events e
    ON e.user_id = s.user_id
   AND CAST(e.event_timestamp AS DATE) BETWEEN DATE'2026-06-21' AND DATE'2026-09-19'
  WHERE s.plan_tier <> 'Free'
    AND s.monthly_price_usd NOT IN (-9.99, 999.99)
  GROUP BY s.subscription_id, s.status, s.start_date, s.monthly_price_usd
) x
GROUP BY 1
ORDER BY 1;

status_group,subscriptions,active_last_90_days,pct_active_last_90_days,listed_monthly_price
TEST_STATUS_DO_NOT_USE,15,13,86.7,165.85
active,311,268,86.2,4087.89
canceled,188,163,86.7,2851.12
paused,69,60,87.0,948.31
recent trial,4,3,75.0,76.96
stale trial,25,22,88.0,353.75


In [0]:
WITH ordered AS (
  SELECT user_id,
         business_id,
         COUNT(*) AS subscriptions,
         ARRAY_JOIN(
           TRANSFORM(ARRAY_SORT(COLLECT_LIST(STRUCT(start_date, subscription_id, status))), x -> x.status),
           ' → ') AS status_path
  FROM workspace.scribd_exercise.exercise_subscriptions
  GROUP BY user_id, business_id
)
SELECT subscriptions,
       status_path,
       COUNT(*) AS user_business_pairs
FROM ordered
GROUP BY subscriptions, status_path
ORDER BY subscriptions DESC, user_business_pairs DESC;

subscriptions,status_path,user_business_pairs
3,active → active → canceled,2
3,active → canceled → canceled,1
3,active → trialing → canceled,1
3,active → active → active,1
2,active → active,23
2,canceled → active,11
2,canceled → canceled,4
2,active → paused,4
2,active → canceled,3
2,active → trialing,2


In [0]:
WITH open_subs AS (
  SELECT *,
         ROW_NUMBER() OVER (PARTITION BY user_id, business_id ORDER BY start_date DESC, subscription_id DESC) AS recency
  FROM workspace.scribd_exercise.exercise_subscriptions
  WHERE status <> 'canceled'
),
pairs AS (
  SELECT user_id,
         business_id,
         COUNT(*) AS open_count,
         COUNT(DISTINCT plan_tier) AS plans
  FROM open_subs
  GROUP BY user_id, business_id
  HAVING COUNT(*) > 1
)
SELECT CASE WHEN p.plans = 1 THEN 'same plan' ELSE 'different plans' END AS plan_pattern,
       COUNT(DISTINCT CONCAT(o.user_id, '|', o.business_id)) AS user_business_pairs,
       COUNT(*) AS open_subscriptions,
       SUM(CASE WHEN o.recency > 1 THEN 1 ELSE 0 END) AS older_open_subscriptions,
       ROUND(SUM(CASE WHEN o.recency > 1 AND o.status = 'active' THEN o.monthly_price_usd ELSE 0 END), 2) AS mrr_from_older_active
FROM open_subs o
JOIN pairs p
  ON o.user_id = p.user_id
 AND o.business_id = p.business_id
GROUP BY 1;

plan_pattern,user_business_pairs,open_subscriptions,older_open_subscriptions,mrr_from_older_active
different plans,30,61,31,230.81
same plan,7,14,7,74.96


**Findings:** 
- 92% of user–business pairs have a single subscription. Status paths (oldest → newest, per business) show trials never lead to a new subscription record ("trialing → active" never appears)
- 11 pairs went "canceled → active" (win-backs)
- 37 pairs have 2+ open subscriptions at once: 30 with different plans (likely plan changes never closed)
- 7 with the same plan (likely duplicates). 
- 38 older open subscriptions add about $306/month of double-counted MRR.

**Treatment:** 
- flag older open subscriptions superseded_open_subscription. 
- Active subscribers = distinct users with ≥1 active subscription per business
- MRR = most recent active subscription per user and business.

In [0]:
-- Follow up on the 7 same-plan overlaps. Are they duplicate subscriptions or renewals? 
WITH open_subs AS (
  SELECT *
  FROM workspace.scribd_exercise.exercise_subscriptions
  WHERE status <> 'canceled'
),
same_plan_pairs AS (
  SELECT user_id, business_id
  FROM open_subs
  GROUP BY user_id, business_id
  HAVING COUNT(*) > 1 AND COUNT(DISTINCT plan_tier) = 1
)
SELECT o.user_id,
       o.business_id,
       o.subscription_id,
       o.plan_tier,
       COALESCE(o.billing_cycle, 'NULL') AS billing_cycle,
       o.monthly_price_usd,
       o.status,
       o.start_date,
       DATEDIFF(o.start_date,
                LAG(o.start_date) OVER (PARTITION BY o.user_id, o.business_id ORDER BY o.start_date)) AS days_since_previous
FROM open_subs o
JOIN same_plan_pairs p
  ON o.user_id = p.user_id
 AND o.business_id = p.business_id
ORDER BY o.user_id, o.business_id, o.start_date;

user_id,business_id,subscription_id,plan_tier,billing_cycle,monthly_price_usd,status,start_date,days_since_previous
USR_00027,BIZ_2,SUB_000028,Free,NULL,0.0,active,2025-12-18,null
USR_00027,BIZ_2,SUB_000027,Free,NULL,0.0,active,2026-06-07,171
USR_00412,BIZ_1,SUB_000435,Premium_Plus_Deluxe_XL,lifetime_deal_fake,14.99,active,2025-12-14,null
USR_00412,BIZ_1,SUB_000434,Premium_Plus_Deluxe_XL,annual,19.99,paused,2026-06-21,189
USR_00428,BIZ_1,SUB_000450,Premium_Plus_Deluxe_XL,monthly,24.99,active,2025-09-09,null
USR_00428,BIZ_1,SUB_000451,Premium_Plus_Deluxe_XL,monthly,24.99,active,2026-04-07,210
USR_00508,BIZ_2,SUB_000536,Premium_Plus_Deluxe_XL,monthly,24.99,active,2023-02-27,null
USR_00508,BIZ_2,SUB_000534,Premium_Plus_Deluxe_XL,annual,19.99,paused,2023-07-09,132
USR_00713,BIZ_1,SUB_000741,Free,NULL,0.0,active,2025-09-14,null
USR_00713,BIZ_1,SUB_000740,Free,NULL,0.0,active,2025-09-18,4


# Engagement Events Table 

### Uniqueness - is each event listed once

In [0]:
SELECT * FROM workspace.scribd_exercise.exercise_engagement_events LIMIT 5;

event_id,user_id,content_id,event_type,event_timestamp,device,session_duration_sec
EVT_0000001,USR_00257,CNT_00250,content_open,2024-11-22T22:36:12.000Z,web_desktop,2076
EVT_0000002,USR_00285,CNT_00093,content_complete,2026-05-11T07:49:08.000Z,iOS_app,3077
EVT_0000003,USR_00630,CNT_00002,content_open,2025-09-08T13:56:33.000Z,web_mobile,1272
EVT_0000004,USR_00152,CNT_00259,content_complete,2026-09-04T04:49:13.000Z,iOS_app,3525
EVT_0000005,USR_00446,CNT_00021,like,2026-01-07T15:57:17.000Z,iOS_app,14030


In [0]:
SELECT COUNT(*) AS total_rows,
       COUNT(DISTINCT event_id) AS distinct_ids,
       COUNT(*) - COUNT(DISTINCT event_id) AS extra_rows
FROM workspace.scribd_exercise.exercise_engagement_events;

total_rows,distinct_ids,extra_rows
6000,6000,0


### Completeness - which columns have missing values? 

In [0]:
-- Missing values in every column
SELECT SUM(CASE WHEN event_id IS NULL OR TRIM(event_id) = '' THEN 1 ELSE 0 END) AS missing_event_id,
       SUM(CASE WHEN user_id IS NULL OR TRIM(user_id) = '' THEN 1 ELSE 0 END) AS missing_user_id,
       SUM(CASE WHEN content_id IS NULL OR TRIM(content_id) = '' THEN 1 ELSE 0 END) AS missing_content_id,
       SUM(CASE WHEN event_type IS NULL OR TRIM(event_type) = '' THEN 1 ELSE 0 END) AS missing_event_type,
       SUM(CASE WHEN event_timestamp IS NULL THEN 1 ELSE 0 END) AS missing_timestamp,
       SUM(CASE WHEN device IS NULL OR TRIM(device) = '' THEN 1 ELSE 0 END) AS missing_device,
       SUM(CASE WHEN session_duration_sec IS NULL THEN 1 ELSE 0 END) AS missing_duration
FROM workspace.scribd_exercise.exercise_engagement_events;

missing_event_id,missing_user_id,missing_content_id,missing_event_type,missing_timestamp,missing_device,missing_duration
0,0,323,0,0,0,0


In [0]:
-- which event types have missing content? 
SELECT event_type,
       COUNT(*) AS events,
       SUM(CASE WHEN content_id IS NULL OR TRIM(content_id) = '' THEN 1 ELSE 0 END) AS without_content,
       ROUND(100.0 * SUM(CASE WHEN content_id IS NULL OR TRIM(content_id) = '' THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_without_content
FROM workspace.scribd_exercise.exercise_engagement_events
GROUP BY event_type
ORDER BY without_content DESC;

event_type,events,without_content,pct_without_content
subscription_cancel,85,85,100.0
subscription_start,84,84,100.0
page_view,76,76,100.0
search_query,76,76,100.0
glitch_in_the_matrix,40,2,5.0
share,556,0,0.0
content_complete,2838,0,0.0
comment_post,549,0,0.0
content_open,593,0,0.0
like,545,0,0.0


In [0]:
--Does every content_id exist?
SELECT COUNT(*) AS events_with_unknown_content,
       COUNT(DISTINCT e.content_id) AS unknown_content_ids
FROM workspace.scribd_exercise.exercise_engagement_events e
LEFT ANTI JOIN workspace.scribd_exercise.exercise_content_items ci
  ON e.content_id = ci.content_id
WHERE e.content_id IS NOT NULL;

events_with_unknown_content,unknown_content_ids
0,0


**Finding:** 
- No missing values except content_id (323). These are fully explained by event type: 100% of subscription_cancel (85), subscription_start (84), page_view (76) and search_query (76) have no content, since they aren't about one piece of content ("not applicable"), plus 2 glitch_in_the_matrix events.
- All content events have a content_id, and every content_id exists in content_items.

**Note:** content_complete (2,838) far exceeds content_open (593), an inverted funnel.

### Valid values

In [0]:
-- What are the glitch_in_the_matrix events? 
WITH u AS (
  SELECT user_id, MAX(CAST(is_test_account_flag AS INT)) AS is_test
  FROM workspace.scribd_exercise.exercise_users
  GROUP BY user_id
)
SELECT COUNT(*) AS events,
       COUNT(DISTINCT e.user_id) AS users,
       SUM(CASE WHEN u.is_test = 1 THEN 1 ELSE 0 END) AS from_test_accounts,
       COLLECT_SET(e.device) AS devices,
       MIN(e.session_duration_sec) AS min_duration,
       MAX(e.session_duration_sec) AS max_duration,
       MIN(e.event_timestamp) AS first_event,
       MAX(e.event_timestamp) AS last_event
FROM workspace.scribd_exercise.exercise_engagement_events e
LEFT JOIN u ON e.user_id = u.user_id
WHERE e.event_type = 'glitch_in_the_matrix';

events,users,from_test_accounts,devices,min_duration,max_duration,first_event,last_event
40,39,2,"List(iOS_app, web_desktop, smart_toaster, Android_app, web_mobile)",-15,4231,2021-11-06T05:06:26.000Z,2026-09-13T05:10:44.000Z


In [0]:
-- Devices in events and users? 
SELECT 'events (device)' AS source, device AS value, COUNT(*) AS n
FROM workspace.scribd_exercise.exercise_engagement_events
GROUP BY device
UNION ALL
SELECT 'users (primary_device)', primary_device, COUNT(DISTINCT user_id)
FROM workspace.scribd_exercise.exercise_users
GROUP BY primary_device
ORDER BY source, n DESC;

source,value,n
events (device),Android_app,1571
events (device),web_mobile,1521
events (device),web_desktop,1457
events (device),iOS_app,1401
events (device),smart_toaster,50
users (primary_device),Android_app,205
users (primary_device),web_mobile,193
users (primary_device),web_desktop,182
users (primary_device),iOS_app,180
users (primary_device),smart_toaster,20


In [0]:
-- The inverted funnel: completeness without opens 
WITH pairs AS (
  SELECT user_id,
         content_id,
         SUM(CASE WHEN event_type = 'content_open' THEN 1 ELSE 0 END) AS opens,
         SUM(CASE WHEN event_type = 'content_complete' THEN 1 ELSE 0 END) AS completes,
         MIN(CASE WHEN event_type = 'content_open' THEN event_timestamp END) AS first_open,
         MIN(CASE WHEN event_type = 'content_complete' THEN event_timestamp END) AS first_complete
  FROM workspace.scribd_exercise.exercise_engagement_events
  WHERE content_id IS NOT NULL
  GROUP BY user_id, content_id
)
SELECT SUM(CASE WHEN completes > 0 THEN 1 ELSE 0 END) AS user_content_pairs_completed,
       SUM(CASE WHEN completes > 0 AND opens = 0 THEN 1 ELSE 0 END) AS completed_never_opened,
       SUM(CASE WHEN completes > 0 AND opens > 0 AND first_complete < first_open THEN 1 ELSE 0 END) AS completed_before_opened,
       SUM(CASE WHEN completes > 1 THEN 1 ELSE 0 END) AS completed_more_than_once,
       SUM(CASE WHEN opens > 0 AND completes = 0 THEN 1 ELSE 0 END) AS opened_not_completed
FROM pairs;

user_content_pairs_completed,completed_never_opened,completed_before_opened,completed_more_than_once,opened_not_completed
2818,2810,3,20,584


In [0]:
-- Compare subscription events with the subscriptions table, per user

WITH users_with_start_event AS (
  SELECT DISTINCT user_id
  FROM workspace.scribd_exercise.exercise_engagement_events
  WHERE event_type = 'subscription_start'
),
users_with_cancel_event AS (
  SELECT DISTINCT user_id
  FROM workspace.scribd_exercise.exercise_engagement_events
  WHERE event_type = 'subscription_cancel'
),
subscribers AS (
  SELECT DISTINCT user_id
  FROM workspace.scribd_exercise.exercise_subscriptions
),
canceled_subscribers AS (
  SELECT DISTINCT user_id
  FROM workspace.scribd_exercise.exercise_subscriptions
  WHERE status = 'canceled'
)

SELECT
  -- Starts
  (SELECT COUNT(*) FROM subscribers) AS users_with_subscriptions,
  (SELECT COUNT(*) FROM subscribers s
     JOIN users_with_start_event e ON s.user_id = e.user_id) AS of_which_with_start_event,
  (SELECT COUNT(*) FROM users_with_start_event e
     LEFT ANTI JOIN subscribers s ON e.user_id = s.user_id) AS start_event_but_no_subscription,

  -- Cancellations
  (SELECT COUNT(*) FROM canceled_subscribers) AS users_with_canceled_sub,
  (SELECT COUNT(*) FROM canceled_subscribers c
     JOIN users_with_cancel_event e ON c.user_id = e.user_id) AS of_which_with_cancel_event,
  (SELECT COUNT(*) FROM users_with_cancel_event e
     LEFT ANTI JOIN canceled_subscribers c ON e.user_id = c.user_id) AS cancel_event_but_no_canceled_sub;-- Compare subscription events with the subscriptions table, per user


users_with_subscriptions,of_which_with_start_event,start_event_but_no_subscription,users_with_canceled_sub,of_which_with_cancel_event,cancel_event_but_no_canceled_sub
672,69,12,209,24,57


**Findings:**
- 40 glitch_in_the_matrix events (0.7%), spread across 39 users, all devices and 2021–2026: a recurring logging problem. Quarantine.
- 50 events (0.8%) with device smart_toaster, as in users. Map to 'Unknown' and flag.
- 99.7% of completed user–content pairs (2,810 of 2,818) have no open event; opens and completes behave as independent signals. No completion rate or funnel will be build
- Subscription events don't match subscriptions: only 10% of subscribed users have a start event 
- 11% of users with a canceled subscription have a cancel event
- 57 users have a cancel event but no canceled subscription. The subscriptions table is the only source of truth for subscriber metrics

### Session durations - are they plausible?

In [0]:
-- Overall distribution and suspicious values 
SELECT COUNT(*) AS events,
       SUM(CASE WHEN session_duration_sec < 0 THEN 1 ELSE 0 END) AS negative,
       SUM(CASE WHEN session_duration_sec = 0 THEN 1 ELSE 0 END) AS zero,
       SUM(CASE WHEN session_duration_sec > 14400 THEN 1 ELSE 0 END) AS over_4_hours,
       SUM(CASE WHEN session_duration_sec = 604800 THEN 1 ELSE 0 END) AS exactly_7_days,
       MIN(session_duration_sec) AS min_sec,
       PERCENTILE(session_duration_sec, 0.25) AS p25,
       PERCENTILE(session_duration_sec, 0.5) AS median,
       PERCENTILE(session_duration_sec, 0.75) AS p75,
       PERCENTILE(session_duration_sec, 0.99) AS p99,
       MAX(session_duration_sec) AS max_sec,
       ROUND(AVG(session_duration_sec), 0) AS avg_sec
FROM workspace.scribd_exercise.exercise_engagement_events;

events,negative,zero,over_4_hours,exactly_7_days,min_sec,p25,median,p75,p99,max_sec,avg_sec
6000,30,0,104,30,-15,1201.0,2990.5,3593.0,40583.270000000004,604800,6008.0


In [0]:
-- The most extreme values 
-- Lowest durations
SELECT session_duration_sec, COUNT(*) AS events
FROM workspace.scribd_exercise.exercise_engagement_events
GROUP BY session_duration_sec
ORDER BY session_duration_sec DESC
LIMIT 10;

session_duration_sec,events
604800,30
86400,4
85496,1
80236,1
80063,1
78152,1
77331,1
74074,1
72284,1
50180,1


**Finding:** 
- Placeholder durations: 30 events at -15 seconds
- 30 at exactly 604,800 (7 days) 
- 4 at exactly 86,400 (1 day). 
- Another 70 events exceed 4 hours (up to ~24 hours) with irregular values: plausible but likely idle sessions.

**Impact:** the 7-day placeholders roughly double the average session (6,008 s ≈ 100 min raw vs. ≈ 3,014 s ≈ 50 min without placeholders); the median is 2,990 s (≈ 50 min).

**Treatment:** keep all events; set placeholder durations to null (invalid_duration); flag sessions over 4 hours (long_session); time-spent metrics use the median.

### Timestamps, freshness and heavy users 

In [0]:
--Date range
SELECT MIN(event_timestamp) AS first_event,
       MAX(event_timestamp) AS last_event,
       SUM(CASE WHEN event_timestamp > CURRENT_TIMESTAMP() THEN 1 ELSE 0 END) AS future_events,
       SUM(CASE WHEN event_timestamp < TIMESTAMP'2021-01-01' THEN 1 ELSE 0 END) AS before_first_launch,
       SUM(CASE WHEN event_timestamp >= TIMESTAMP'2026-09-20' THEN 1 ELSE 0 END) AS after_other_tables_end
FROM workspace.scribd_exercise.exercise_engagement_events;

first_event,last_event,future_events,before_first_launch,after_other_tables_end
2021-01-09T07:58:18.000Z,2026-10-12T17:32:27.000Z,0,0,5


In [0]:
-- Events and active users per month
SELECT DATE_TRUNC('month', event_timestamp) AS month,
       COUNT(*) AS events,
       COUNT(DISTINCT user_id) AS active_users
FROM workspace.scribd_exercise.exercise_engagement_events
GROUP BY DATE_TRUNC('month', event_timestamp)
ORDER BY month;

month,events,active_users
2021-01-01T00:00:00.000Z,1,1
2021-03-01T00:00:00.000Z,1,1
2021-04-01T00:00:00.000Z,2,2
2021-05-01T00:00:00.000Z,4,4
2021-08-01T00:00:00.000Z,1,1
2021-09-01T00:00:00.000Z,4,4
2021-10-01T00:00:00.000Z,1,1
2021-11-01T00:00:00.000Z,2,2
2022-01-01T00:00:00.000Z,4,4
2022-02-01T00:00:00.000Z,1,1


In [0]:
-- Heavy users 
-- How many events do users typically have?
WITH per_user AS (
  SELECT user_id,
         COUNT(*) AS events,
         COUNT(DISTINCT CAST(event_timestamp AS DATE)) AS active_days
  FROM workspace.scribd_exercise.exercise_engagement_events
  GROUP BY user_id
)
SELECT COUNT(*) AS users,
       MIN(events) AS min_events,
       PERCENTILE(events, 0.5) AS median_events,
       PERCENTILE(events, 0.99) AS p99_events,
       MAX(events) AS max_events,
       MAX(active_days) AS max_active_days
FROM per_user;

users,min_events,median_events,p99_events,max_events,max_active_days
799,1,7.0,14.0,16,16


In [0]:
-- The 10 most active users
SELECT e.user_id,
       COUNT(*) AS events,
       COUNT(DISTINCT CAST(e.event_timestamp AS DATE)) AS active_days,
       MAX(CAST(u.is_test_account_flag AS INT)) AS is_test
FROM workspace.scribd_exercise.exercise_engagement_events e
LEFT JOIN workspace.scribd_exercise.exercise_users u
  ON e.user_id = u.user_id
GROUP BY e.user_id
ORDER BY events DESC
LIMIT 10;

user_id,events,active_days,is_test
USR_00622,20,9,0
USR_00064,18,9,0
USR_00067,16,3,0
USR_00554,16,15,0
USR_00191,16,16,0
USR_00651,15,15,0
USR_00062,15,15,0
USR_00075,14,14,0
USR_00588,14,14,0
USR_00668,14,6,0


**Findings** 
-  September is incomplete everywhere. Reporting month: August 2026
- Raw monthly active users grew steadily from 1–5 in 2021 to 431 in August 2026, with activity every month since January 2022.
- No bot-like users: median 7 events per user, max 20, spread over many days.